# 🐼 Pandas — полный конспект

> **Версия:** 2026 · **Уровень:** от основ до продвинутых тем · **Разделов:** 37

pandas — стандарт индустрии для работы с табличными данными в Python. Этот конспект охватывает всё: от создания первого DataFrame до оптимизации памяти и pipeline-анализа.

---

## Оглавление

| № | Тема |
|---|------|
| 1 | Что такое pandas |
| 2 | Импорт |
| 3 | Основные структуры данных |
| 4 | Создание DataFrame |
| 5 | Просмотр данных |
| 6 | Чтение данных |
| 7 | Запись данных |
| 8 | Выбор колонок |
| 9 | Выбор строк и фильтрация |
| 10 | Индексы |
| 11 | Типы данных |
| 12 | Пропуски |
| 13 | Дубликаты |
| 14 | Сортировка |
| 15 | Базовые расчёты |
| 16 | Условные колонки |
| 17 | apply и map |
| 18 | Работа со строками (.str) |
| 19 | Группировка groupby |
| 20 | Сводные таблицы |
| 21 | Объединение таблиц |
| 22 | Изменение формы таблицы |
| 23 | Даты и время |
| 24 | Временные ряды |
| 25 | Оконные функции |
| 26 | Ранжирование и накопительные расчёты |
| 27 | MultiIndex |
| 28 | Категориальные данные |
| 29 | assign и pipe |
| 30 | explode |
| 31 | Работа с NumPy |
| 32 | Производительность и память |
| 33 | Copy-on-Write |
| 34 | Styler |
| 35 | Частые ошибки |
| 36 | Бизнес-примеры |
| 37 | Шаблон анализа + Шпаргалка |


---
## 1. Что такое pandas

### Краткая история

pandas создал **Уэс МакКинни** в 2008 году, работая финансовым аналитиком. Инструменты того времени (Excel, R, SQL) не позволяли гибко работать с большими объёмами данных прямо в коде. Так появился pandas — **pan**el **da**ta **s**ystem.

Сегодня pandas входит в стандартный стек Data Science наравне с NumPy, Matplotlib и scikit-learn.

### Где применяется

| Область | Примеры задач |
|---------|--------------|
| **Бизнес-аналитика** | план-факт, воронки продаж, сегментация клиентов |
| **Маркетинг** | CTR, CPC, ROMI, когортный анализ |
| **Финансы** | P&L, дебиторская задолженность, бюджетирование |
| **Data Science / ML** | очистка и подготовка данных перед обучением |
| **Аналитика продукта** | DAU/MAU, retention, A/B тесты |
| **Операционный учёт** | обработка выгрузок из ERP, CRM, BI-систем |

### Отношение к NumPy

pandas построен **поверх NumPy**. Внутри каждая колонка DataFrame — это NumPy-массив. Но pandas добавляет:

- **Именованные колонки** — не `arr[:, 2]`, а `df["revenue"]`
- **Индексы строк** — метки вместо позиций
- **Гетерогенные типы** — разные колонки могут быть разных типов
- **Работу с пропусками** — `NaN` обрабатывается автоматически
- **Богатый API** — группировки, объединения, временные ряды, pivot

> **Правило:** если данные — числовой однотипный массив, используй NumPy. Если данные — таблица с разными колонками и метками, используй pandas.


### Философия pandas: «think in vectors»

Главная мысль, которую нужно усвоить при переходе на pandas — **думать векторами, а не циклами**. Вместо того чтобы перебирать строки таблицы (как в Excel или VBA), pandas заставляет работать со всей колонкой сразу.

```python
# ❌ Мышление в стиле "циклов" (медленно, не идиоматично):
for i in range(len(df)):
    df.at[i, "tax"] = df.at[i, "price"] * 0.2

# ✅ Векторное мышление (быстро, идиоматично):
df["tax"] = df["price"] * 0.2
```

Это требует перестройки привычек, но даёт огромный выигрыш в скорости (часто в **100–1000 раз**) и в читаемости кода.

### Какие задачи решает pandas «из коробки»

| Задача | Метод pandas |
|--------|-------------|
| Прочитать CSV/Excel/JSON | `pd.read_csv()`, `pd.read_excel()` |
| Отфильтровать строки | булева маска `df[df["col"] > 0]` |
| Сгруппировать и посчитать | `df.groupby(...).agg(...)` |
| Объединить таблицы (JOIN) | `df.merge(...)` |
| Сводная таблица | `pd.pivot_table(...)` |
| Заполнить пропуски | `df.fillna(...)` |
| Временные ряды | `df.resample(...)`, `df.rolling(...)` |
| Сохранить результат | `df.to_csv(...)`, `df.to_excel(...)` |


---
## 2. Импорт pandas

По соглашению pandas всегда импортируется с псевдонимом `pd` — это мировой стандарт. Никогда не делайте `import pandas` без псевдонима: код станет нечитаемым, и коллеги не поймут.


In [135]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("numpy: ", np.__version__)

pandas: 2.3.3
numpy:  2.0.2


> **Совет по версиям:** рекомендуется pandas ≥ 2.0. В pandas 3.0 по умолчанию включён Copy-on-Write и изменился тип строк. Некоторые старые паттерны (`inplace=True`, цепочные присваивания) дают предупреждения или ведут себя иначе.


---
## 3. Основные структуры данных

В pandas две ключевые структуры: **Series** и **DataFrame**. Понимание разницы между ними — фундамент работы с библиотекой.

```
DataFrame (таблица)
┌─────────────┬──────────┬────────┐
│  client_id  │ revenue  │ orders │
├─────────────┼──────────┼────────┤
│     101     │  100 000 │   2    │  ← строка (row)
│     102     │  250 000 │   5    │
│     103     │  175 000 │   3    │
└─────────────┴──────────┴────────┘
       ↑           ↑
    колонка = Series
```

### 3.1 Series — одномерная структура

Series можно воспринимать как **именованный массив** или **одну колонку таблицы**. У каждого элемента есть:
- **значение** — сами данные
- **метка индекса** — адрес элемента (по умолчанию 0, 1, 2, ...)

Series поддерживает все NumPy-операции: векторная арифметика, логические маски, агрегации. При этом pandas автоматически **выравнивает Series по индексу** при операциях — это избавляет от ошибок при разных порядках строк.


### 3.0 Выравнивание по индексу (alignment) — невидимая магия pandas

Одна из самых важных и недооценённых фич pandas — **автоматическое выравнивание по индексу** при арифметике. Это значит, что когда вы складываете две Series, pandas сначала «выстраивает» их по совпадающим меткам, а потом складывает.

Это поведение похоже на **LEFT JOIN на лету**: где меток нет — ставится `NaN`. Это защищает от ошибок при работе с данными, упорядоченными по-разному.


In [136]:
import numpy as np
import pandas as pd

# Две Series с РАЗНЫМ порядком индексов
s1 = pd.Series([100, 200, 300], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30],    index=["c", "a", "b"])  # порядок другой!

# При сложении pandas сам выравнивает по меткам
result = s1 + s2
print(result)
# a -> 100 + 20 = 120
# b -> 200 + 30 = 230
# c -> 300 + 10 = 310

a    120
b    230
c    310
dtype: int64


In [137]:
# Что происходит, когда меток в одной Series нет — NaN
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20],  index=["b", "d"])

print(s1 + s2)
# a -> NaN  (нет в s2)
# b ->   12 (есть в обеих)
# c -> NaN  (нет в s2)
# d -> NaN  (нет в s1)

# Если нужно заполнить NaN — используем .add() с fill_value
print(s1.add(s2, fill_value=0))

a     NaN
b    12.0
c     NaN
d     NaN
dtype: float64
a     1.0
b    12.0
c     3.0
d    20.0
dtype: float64


### Бродкастинг (broadcasting) — арифметика DataFrame и Series

При операциях `DataFrame ± Series` происходит **бродкастинг**: Series «расширяется» до размера DataFrame. По умолчанию выравнивание идёт по колонкам (axis=1), но это можно изменить.


In [138]:
# Пример: вычесть среднее по каждой колонке из всех строк
df = pd.DataFrame({
    "a": [10, 20, 30],
    "b": [100, 200, 300],
    "c": [1000, 2000, 3000],
})

means = df.mean()   # Series со средним по каждой колонке
print("Средние:")
print(means)

# Бродкастинг: каждая колонка минус своё среднее
centered = df - means
print("\nЦентрированный df:")
centered

Средние:
a      20.0
b     200.0
c    2000.0
dtype: float64

Центрированный df:


,a,b,c
0,-10.0,-100.0,-1000.0
1,0.0,0.0,0.0
2,10.0,100.0,1000.0


In [139]:
# Создание Series из списка
s = pd.Series([10, 20, 30, 40])
print(s)
print()
print("Тип:     ", type(s))
print("dtype:   ", s.dtype)
print("Значения:", s.values)
print("Индекс:  ", s.index.tolist())

0    10
1    20
2    30
3    40
dtype: int64

Тип:      <class 'pandas.core.series.Series'>
dtype:    int64
Значения: [10 20 30 40]
Индекс:   [0, 1, 2, 3]


In [140]:
# Series с именованным индексом
s_named = pd.Series(
    [100_000, 250_000, 175_000],
    index=["Москва", "СПб", "Казань"],
    name="revenue_rub"
)
print(s_named)
print()
print("По метке:", s_named["Москва"])
print("По позиции:", s_named.iloc[0])
print("Срез:")
print(s_named["Москва":"СПб"])

Москва    100000
СПб       250000
Казань    175000
Name: revenue_rub, dtype: int64

По метке: 100000
По позиции: 100000
Срез:
Москва    100000
СПб       250000
Name: revenue_rub, dtype: int64


In [141]:
# Series из словаря — ключи становятся индексом
sales = pd.Series({"Иванов": 1_200_000, "Петров": 980_000, "Сидорова": 1_450_000})
print(sales)
print()
# Векторная арифметика — операции применяются поэлементно
print("С бонусом 10%:")
print((sales * 1.1).round(0))

Иванов      1200000
Петров       980000
Сидорова    1450000
dtype: int64

С бонусом 10%:
Иванов      1320000.0
Петров      1078000.0
Сидорова    1595000.0
dtype: float64


### 3.2 DataFrame — двумерная таблица

DataFrame — это **таблица с именованными строками и колонками**. Внутри это словарь Series: каждая колонка — отдельная Series с общим индексом. Именно поэтому все колонки одной таблицы имеют одинаковое количество строк.

Ключевые атрибуты DataFrame:

| Атрибут | Что хранит |
|---------|-----------|
| `.index` | метки строк |
| `.columns` | названия колонок |
| `.values` | данные как NumPy-массив |
| `.dtypes` | типы данных по колонкам |
| `.shape` | кортеж `(строк, колонок)` |


In [142]:
df = pd.DataFrame({
    "client_id": [101, 102, 103],
    "revenue":   [100_000, 250_000, 175_000],
    "orders":    [2, 5, 3]
})
print(df)
print()
print("shape:  ", df.shape)
print("columns:", df.columns.tolist())
print("index:  ", df.index.tolist())
print()
print("dtypes:")
print(df.dtypes)

   client_id  revenue  orders
0        101   100000       2
1        102   250000       5
2        103   175000       3

shape:   (3, 3)
columns: ['client_id', 'revenue', 'orders']
index:   [0, 1, 2]

dtypes:
client_id    int64
revenue      int64
orders       int64
dtype: object


In [143]:
# Каждая колонка DataFrame — это Series с общим индексом
col = df["revenue"]
print(type(col))    # <class 'pandas.core.series.Series'>
print(col)

<class 'pandas.core.series.Series'>
0    100000
1    250000
2    175000
Name: revenue, dtype: int64


---
## 4. Создание DataFrame

DataFrame можно создать из разных источников. Самый частый способ в аналитике — из словаря списков, где **ключи = названия колонок**, а **списки = данные по колонкам**. Все списки должны быть одной длины.


### 4.1 Из словаря списков

In [144]:
df = pd.DataFrame({
    "manager": ["Иванов", "Петров", "Сидорова"],
    "sales":   [1_000_000, 1_200_000, 900_000],
    "plan":    [1_100_000, 1_000_000, 950_000]
})
df

,manager,sales,plan
0,Иванов,1000000,1100000
1,Петров,1200000,1000000
2,Сидорова,900000,950000


### 4.2 Из списка словарей

Удобно, когда данные приходят как набор записей (из JSON API, базы данных). Каждый словарь — одна строка. Если у словарей разные ключи — недостающие значения заполняются `NaN`.


In [145]:
df = pd.DataFrame([
    {"manager": "Иванов",   "sales": 1_000_000, "region": "Москва"},
    {"manager": "Петров",   "sales": 1_200_000},              # region -> NaN
    {"manager": "Сидорова", "sales":   900_000, "region": "СПб"},
])
df

,manager,sales,region
0,Иванов,1000000,Москва
1,Петров,1200000,NaN
2,Сидорова,900000,СПб


### 4.3 Из NumPy-массива

In [146]:
arr = np.array([[101, 100_000, 2], [102, 250_000, 5], [103, 175_000, 3]])
df = pd.DataFrame(arr, columns=["client_id", "revenue", "orders"])
df

,client_id,revenue,orders
0,101,100000,2
1,102,250000,5
2,103,175000,3


### 4.4 Генерация тестовых данных

В реальных задачах часто нужно быстро создать DataFrame для проверки кода. `np.random` и `pd.date_range` — основные инструменты.


In [147]:
np.random.seed(42)

df = pd.DataFrame({
    "date":    pd.date_range("2026-01-01", periods=20, freq="D"),
    "manager": np.random.choice(["Иванов", "Петров", "Сидорова"], 20),
    "region":  np.random.choice(["Москва", "СПб", "Казань"], 20),
    "revenue": np.random.randint(100_000, 2_000_000, 20),
    "cost":    np.random.randint(50_000, 1_000_000, 20),
    "orders":  np.random.randint(1, 20, 20),
})
df["margin"] = df["revenue"] - df["cost"]
print(f"Создан DataFrame: {df.shape[0]} строк × {df.shape[1]} колонок")
df.head(5)

Создан DataFrame: 20 строк × 7 колонок


,date,manager,region,revenue,cost,orders,margin
0,2026-01-01,Сидорова,Москва,431236,728843,18,-297607
1,2026-01-02,Иванов,Москва,628178,960790,12,-332612
2,2026-01-03,Сидорова,СПб,665894,622843,2,43051
3,2026-01-04,Сидорова,СПб,589492,306508,10,282984
4,2026-01-05,Иванов,Москва,1405416,853591,4,551825


---
## 5. Просмотр данных

Первое, что делают при получении нового DataFrame — изучают его структуру. Это обязательный шаг: он помогает понять, с чем предстоит работать, и избежать ошибок на следующих этапах.

### Стандартный порядок первичного осмотра

1. `head()` — посмотреть первые строки, понять структуру
2. `info()` — типы данных, пропуски, память
3. `describe()` — диапазоны значений, выбросы
4. `isna().sum()` — пропуски по колонкам
5. `duplicated().sum()` — дубликаты

Не пропускайте этот этап. Большинство ошибок в анализе возникает из-за непонимания структуры данных: неверный тип, скрытые пропуски, неожиданные дубли.


### 5.0 Алгоритм первичного осмотра — что именно искать

Первичный осмотр данных — это **диагностика**, а не формальность. Каждый метод отвечает на конкретный вопрос:

| Метод | Что покажет | На что смотреть |
|-------|-------------|-----------------|
| `df.shape` | размер | соответствует ли ожиданиям? |
| `df.head()` | первые строки | правильно ли распарсилось? |
| `df.dtypes` | типы | даты — это `datetime64`, а не строки? |
| `df.info()` | пропуски + память | где `Non-Null Count < total`? |
| `df.describe()` | статистика | странные `min`/`max`? разрыв `mean` vs `median`? |
| `df.isna().sum()` | пропуски | где их слишком много? |
| `df.duplicated().sum()` | дубли | сколько повторов? |
| `df.nunique()` | уникальность | колонка с 1 уникальным значением = бесполезна |

### Красные флаги, которые сигнализируют о проблемах

- `dtype: object` у колонки с датами → даты не распарсены
- `min` = 0 у колонки с ценой → есть нулевые или пропущенные транзакции
- разрыв между `mean` и `50%` (медиана) → распределение скошено, есть выбросы
- `count` в `describe()` меньше `df.shape[0]` → есть пропуски
- `unique()` = 1 у колонки → колонка не несёт информации


In [148]:
df.head()       # первые 5 строк по умолчанию

,date,manager,region,revenue,cost,orders,margin
0,2026-01-01,Сидорова,Москва,431236,728843,18,-297607
1,2026-01-02,Иванов,Москва,628178,960790,12,-332612
2,2026-01-03,Сидорова,СПб,665894,622843,2,43051
3,2026-01-04,Сидорова,СПб,589492,306508,10,282984
4,2026-01-05,Иванов,Москва,1405416,853591,4,551825


In [149]:
df.tail(3)      # последние N строк

,date,manager,region,revenue,cost,orders,margin
17,2026-01-18,Петров,Казань,748531,221829,15,526702
18,2026-01-19,Петров,Москва,1400571,321836,13,1078735
19,2026-01-20,Петров,Казань,1006606,488974,1,517632


In [150]:
df.sample(5, random_state=42)  # случайные строки — полезно для больших датасетов

,date,manager,region,revenue,cost,orders,margin
0,2026-01-01,Сидорова,Москва,431236,728843,18,-297607
17,2026-01-18,Петров,Казань,748531,221829,15,526702
15,2026-01-16,Иванов,СПб,168148,391097,13,-222949
1,2026-01-02,Иванов,Москва,628178,960790,12,-332612
8,2026-01-09,Сидорова,Казань,1419512,872352,8,547160


In [151]:
df.shape        # кортеж (строк, колонок)

(20, 7)

In [152]:
df.columns.tolist()   # список колонок

['date', 'manager', 'region', 'revenue', 'cost', 'orders', 'margin']

In [153]:
df.dtypes             # тип каждой колонки

date       datetime64[ns]
manager            object
region             object
revenue             int64
cost                int64
orders              int64
margin              int64
dtype: object

`df.info()` — один из самых полезных методов при первичном осмотре. Показывает сразу:
- общее количество строк
- название и тип каждой колонки
- количество **непустых** значений (Non-Null Count) — сразу видны пропуски
- примерный объём памяти

Если `Non-Null Count` меньше числа строк — в этой колонке есть пропуски.


In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     20 non-null     datetime64[ns]
 1   manager  20 non-null     object        
 2   region   20 non-null     object        
 3   revenue  20 non-null     int64         
 4   cost     20 non-null     int64         
 5   orders   20 non-null     int64         
 6   margin   20 non-null     int64         
dtypes: datetime64[ns](1), int64(4), object(2)
memory usage: 1.2+ KB


`df.describe()` вычисляет описательную статистику для числовых колонок:

| Метрика | Значение |
|---------|---------|
| `count` | количество непустых значений |
| `mean` | среднее |
| `std` | стандартное отклонение (разброс) |
| `min` | минимум |
| `25%` | первый квартиль |
| `50%` | медиана |
| `75%` | третий квартиль |
| `max` | максимум |

Смотрите на `min`/`max` — они сразу выявляют аномалии (отрицательные выручки, нереальные значения). Сравнивайте `mean` и `median` — большой разрыв означает наличие выбросов.


In [155]:
df.describe()

,date,revenue,cost,orders,margin
count,20,2.000000e+01,20.000000,20.000000,2.000000e+01
mean,2026-01-10 12:00:00,1.027641e+06,580760.350000,11.700000,4.468803e+05
min,2026-01-01 00:00:00,1.681480e+05,156530.000000,1.000000,-3.736880e+05
25%,2026-01-05 18:00:00,6.185065e+05,354313.250000,8.750000,9.447925e+04
50%,2026-01-10 12:00:00,9.024835e+05,566590.000000,13.000000,5.076800e+05
75%,2026-01-15 06:00:00,1.401782e+06,824249.500000,15.000000,6.478295e+05
max,2026-01-20 00:00:00,1.925665e+06,960790.000000,18.000000,1.622227e+06
std,NaN,5.100108e+05,260669.798026,4.974673,5.618222e+05


In [156]:
df.describe(include="all")   # включая нечисловые колонки

,date,manager,region,revenue,cost,orders,margin
count,20,20,20,2.000000e+01,20.000000,20.000000,2.000000e+01
unique,NaN,3,3,NaN,NaN,NaN,NaN
top,NaN,Сидорова,Казань,NaN,NaN,NaN,NaN
freq,NaN,9,8,NaN,NaN,NaN,NaN
mean,2026-01-10 12:00:00,NaN,NaN,1.027641e+06,580760.350000,11.700000,4.468803e+05
min,2026-01-01 00:00:00,NaN,NaN,1.681480e+05,156530.000000,1.000000,-3.736880e+05
25%,2026-01-05 18:00:00,NaN,NaN,6.185065e+05,354313.250000,8.750000,9.447925e+04
50%,2026-01-10 12:00:00,NaN,NaN,9.024835e+05,566590.000000,13.000000,5.076800e+05
75%,2026-01-15 06:00:00,NaN,NaN,1.401782e+06,824249.500000,15.000000,6.478295e+05
max,2026-01-20 00:00:00,NaN,NaN,1.925665e+06,960790.000000,18.000000,1.622227e+06


In [157]:
# Разведка конкретной колонки
print("Уникальных значений:", df["manager"].nunique())
print("Список уникальных:  ", df["manager"].unique())
print()
print("Частота значений:")
print(df["manager"].value_counts())

Уникальных значений: 3
Список уникальных:   ['Сидорова' 'Иванов' 'Петров']

Частота значений:
manager
Сидорова    9
Петров      6
Иванов      5
Name: count, dtype: int64


### 5.10 Дополнительные инструменты разведки

Помимо `head`, `info`, `describe`, есть менее известные, но полезные методы.


In [158]:
# nunique() для всего DataFrame — сколько уникальных значений в каждой колонке
df.nunique()

date       20
manager     3
region      3
revenue    20
cost       20
orders     12
margin     20
dtype: int64

In [159]:
# memory_usage() — память по колонкам (важно при работе с большими данными)
df.memory_usage(deep=True)
# deep=True — считает реальный размер строк, а не указателей

Index       128
date        160
manager    2212
region     2068
revenue     160
cost        160
orders      160
margin      160
dtype: int64

In [160]:
# select_dtypes — выбрать колонки определённых типов
df.select_dtypes(include="number").head()     # только числовые
# df.select_dtypes(include="object")          # только строки
# df.select_dtypes(exclude="number")          # всё, кроме чисел

,revenue,cost,orders,margin
0,431236,728843,18,-297607
1,628178,960790,12,-332612
2,665894,622843,2,43051
3,589492,306508,10,282984
4,1405416,853591,4,551825


In [161]:
# axes — оси (index и columns одним вызовом)
print(df.axes)

# size — общее число элементов (строк × колонок)
print("Всего ячеек:", df.size)

# empty — пустой ли DataFrame
print("Пустой?", df.empty)

[RangeIndex(start=0, stop=20, step=1), Index(['date', 'manager', 'region', 'revenue', 'cost', 'orders', 'margin'], dtype='object')]
Всего ячеек: 140
Пустой? False


---
## 6. Чтение данных

pandas умеет читать десятки форматов. Функции чтения начинаются с `pd.read_*` и все возвращают DataFrame.

### Форматы и их особенности

| Формат | Функция | Когда использовать |
|--------|---------|-------------------|
| CSV | `pd.read_csv()` | универсальный текстовый формат |
| Excel | `pd.read_excel()` | файлы от коллег и клиентов |
| Parquet | `pd.read_parquet()` | большие данные, хранилища (S3, HDFS) |
| JSON | `pd.read_json()` | API-ответы, NoSQL-выгрузки |
| SQL | `pd.read_sql()` | прямой запрос к БД |

### 6.1 CSV — самый частый формат

CSV (Comma-Separated Values) — текстовый формат, где колонки разделены символом. В России часто используется `;` в качестве разделителя, поскольку запятая используется как десятичный разделитель.


In [162]:
# Базовое чтение
# df = pd.read_csv("file.csv")

# Разделитель ; — типично для Excel-экспортов из русской локали
# df = pd.read_csv("file.csv", sep=";")

# Кодировка cp1251 / windows-1251 — стандарт для старых русских файлов
# df = pd.read_csv("file.csv", sep=";", encoding="cp1251")

# Только нужные колонки — сильно экономит память на больших файлах
# df = pd.read_csv("file.csv", usecols=["client_id", "date", "amount"])

# Указать тип данных при чтении (быстрее и экономит память)
# df = pd.read_csv("file.csv", dtype={"client_id": "int32", "region": "category"})

# Пропустить строки
# df = pd.read_csv("file.csv", skiprows=2)       # пропустить первые 2 строки
# df = pd.read_csv("file.csv", nrows=1000)        # прочитать только 1000 строк

# Заменить значения на NaN при чтении
# df = pd.read_csv("file.csv", na_values=["N/A", "-", "нет данных", ""])

print("Ключевые параметры: sep, encoding, usecols, dtype, nrows, na_values")

Ключевые параметры: sep, encoding, usecols, dtype, nrows, na_values


### 6.2 Excel

Для работы с Excel нужна библиотека `openpyxl`: `pip install openpyxl`.


In [163]:
# Базовое чтение первого листа
# df = pd.read_excel("file.xlsx")

# Конкретный лист
# df = pd.read_excel("file.xlsx", sheet_name="Продажи")

# Все листы -> словарь {имя_листа: DataFrame}
# sheets = pd.read_excel("file.xlsx", sheet_name=None)
# df_sales = sheets["Продажи"]
# df_plan  = sheets["План"]

print("sheet_name=None читает все листы -> dict")

sheet_name=None читает все листы -> dict


### 6.3 Parquet — оптимальный формат для больших данных

Parquet — **колоночный бинарный формат** с компрессией. По сравнению с CSV:
- файл в 5–10 раз меньше
- чтение в 5–20 раз быстрее
- **сохраняет типы данных** (CSV всё читает как строки)
- поддерживает чтение только нужных колонок (column pruning)

Используется в облачных хранилищах (S3, GCS), Apache Spark, аналитических БД.


In [164]:
# pip install pyarrow
# df = pd.read_parquet("file.parquet")
# df = pd.read_parquet("file.parquet", columns=["client_id", "revenue"])

print("Parquet — лучший выбор для хранения больших таблиц в production")

Parquet — лучший выбор для хранения больших таблиц в production


### 6.4 Чтение больших файлов по частям (chunking)

Если файл не помещается в RAM, `chunksize` позволяет читать данные **итератором по кускам**. Это классический ETL-подход.


In [165]:
# Обработка файла по 100 000 строк за раз
# chunks = pd.read_csv("large_file.csv", chunksize=100_000)
# results = []
# for chunk in chunks:
#     agg = chunk.groupby("region")["revenue"].sum()
#     results.append(agg)
# final = pd.concat(results).groupby(level=0).sum()

print("chunksize спасает при работе с файлами > объёма RAM")

chunksize спасает при работе с файлами > объёма RAM


---
## 7. Запись данных

Методы записи начинаются с `df.to_*`. Почти всегда нужен параметр **`index=False`** — он не записывает числовой индекс в файл. Без него в CSV/Excel появится лишняя первая колонка с числами 0, 1, 2...


In [166]:
# CSV
# df.to_csv("result.csv", index=False)
# df.to_csv("result.csv", sep=";", index=False, encoding="utf-8-sig")  # BOM для корректного открытия в Excel

# Excel
# df.to_excel("result.xlsx", index=False)
# df.to_excel("result.xlsx", sheet_name="Данные", index=False)

# Несколько листов — ExcelWriter как контекстный менеджер
# with pd.ExcelWriter("report.xlsx", engine="openpyxl") as writer:
#     df_sales.to_excel(writer, sheet_name="Продажи", index=False)
#     df_plan.to_excel(writer,  sheet_name="План",    index=False)
#     df_kpi.to_excel(writer,   sheet_name="KPI",     index=False)

# Parquet
# df.to_parquet("result.parquet", index=False)

# JSON (orient="records" -> список словарей)
# df.to_json("result.json", orient="records", force_ascii=False, indent=2)

print("Правило: почти всегда пишите index=False")

Правило: почти всегда пишите index=False


---
## 8. Выбор колонок

### 8.1 Одна колонка vs несколько — частый источник ошибок

Запомните правило:

| Синтаксис | Результат | Когда использовать |
|-----------|-----------|-------------------|
| `df["col"]` | **Series** | нужна одна колонка для вычислений |
| `df[["col"]]` | **DataFrame** | одна колонка, но нужен DataFrame |
| `df[["a","b","c"]]` | **DataFrame** | несколько колонок |

Двойные скобки `[[...]]` — это просто `df[list]`, где список содержит имена колонок.


### 8.0 Концептуально: колонки vs строки

Запомните принципиальное правило: **обращение в квадратных скобках без `.loc`/`.iloc` адресует колонки**.

```python
df["revenue"]         # колонка revenue
df[["a", "b"]]        # колонки a и b
df[df["a"] > 0]       # ИСКЛЮЧЕНИЕ: если внутри булева маска -> фильтрует строки
df[0:5]               # ИСКЛЮЧЕНИЕ: если срез -> срез по строкам
```

Чтобы избежать путаницы — используйте `.loc[строки, колонки]` для явной адресации.


In [167]:
revenue_series = df["revenue"]
print(type(revenue_series), revenue_series.shape)
print(revenue_series.head())

<class 'pandas.core.series.Series'> (20,)
0     431236
1     628178
2     665894
3     589492
4    1405416
Name: revenue, dtype: int64


In [168]:
df_sub = df[["manager", "revenue", "margin"]]
print(type(df_sub), df_sub.shape)
df_sub.head()

<class 'pandas.core.frame.DataFrame'> (20, 3)


,manager,revenue,margin
0,Сидорова,431236,-297607
1,Иванов,628178,-332612
2,Сидорова,665894,43051
3,Сидорова,589492,282984
4,Иванов,1405416,551825


### 8.2 Создание новых колонок

Новые колонки создаются присваиванием `df["new_col"] = выражение`. Выражение вычисляется **векторно** — сразу по всей колонке. Это быстро и не требует циклов `for`.


In [169]:
df["margin"]      = df["revenue"] - df["cost"]
df["margin_rate"] = (df["margin"] / df["revenue"]).round(4)
df["currency"]    = "RUB"          # константа
df["is_profitable"] = df["margin"] > 0   # булево

df[["manager","revenue","cost","margin","margin_rate"]].head()

,manager,revenue,cost,margin,margin_rate
0,Сидорова,431236,728843,-297607,-0.6901
1,Иванов,628178,960790,-332612,-0.5295
2,Сидорова,665894,622843,43051,0.0647
3,Сидорова,589492,306508,282984,0.4800
4,Иванов,1405416,853591,551825,0.3926


### 8.3 Переименование колонок

`rename()` — безопасный способ переименовать колонки через словарь `{старое: новое}`. Метод **не меняет исходный DataFrame** (возвращает новый).


In [170]:
df = df.rename(columns={"revenue": "revenue_rub", "cost": "cost_rub"})
df = df.rename(columns={"revenue_rub": "revenue", "cost_rub": "cost"})  # обратно

# Привести все названия к нижнему регистру и заменить пробелы на _
# df.columns = df.columns.str.lower().str.replace(" ", "_")

df.columns.tolist()

['date',
 'manager',
 'region',
 'revenue',
 'cost',
 'orders',
 'margin',
 'margin_rate',
 'currency',
 'is_profitable']

### 8.4 Удаление колонок

`drop(columns=[...])` удаляет колонки. Всегда указывайте `columns=` явно — это защищает от случайного удаления строк.


In [171]:
df_tmp = df.drop(columns=["currency", "is_profitable"])
print("Оставшиеся колонки:", df_tmp.columns.tolist())

Оставшиеся колонки: ['date', 'manager', 'region', 'revenue', 'cost', 'orders', 'margin', 'margin_rate']


### 8.5 Изменение порядка колонок

In [172]:
desired = ["date","manager","region","revenue","cost","margin","margin_rate","orders"]
df = df[[c for c in desired if c in df.columns]]
# [добавь c в новый список for каждый c из desired if c есть в df.columns]
df.head(3)

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2


### 8.6 Дополнительные инструменты для работы с колонками

#### df.filter() — выбор колонок по шаблону имени

`filter()` особенно полезен, когда колонок много и они следуют паттерну именования (например, `revenue_q1`, `revenue_q2`, `revenue_q3`, `revenue_q4`).


In [173]:
# Создадим DataFrame с колонками по паттерну
import numpy as np
import pandas as pd

np.random.seed(0)
df_q = pd.DataFrame({
    "manager":    ["Иванов","Петров","Сидорова"],
    "revenue_q1": np.random.randint(100, 500, 3),
    "revenue_q2": np.random.randint(100, 500, 3),
    "revenue_q3": np.random.randint(100, 500, 3),
    "revenue_q4": np.random.randint(100, 500, 3),
    "cost_q1":    np.random.randint(50, 300, 3),
    "cost_q2":    np.random.randint(50, 300, 3),
})

# Все колонки, содержащие "revenue"
df_q.filter(like="revenue")

,revenue_q1,revenue_q2,revenue_q3,revenue_q4
0,272,292,295,311
1,147,423,459,377
2,217,351,109,342


In [174]:
# Колонки по регулярному выражению
df_q.filter(regex=r"_q[12]$")     # только q1 и q2 любой колонки

# Точный список колонок
df_q.filter(items=["manager", "revenue_q1", "cost_q1"])

,manager,revenue_q1,cost_q1
0,Иванов,272,86
1,Петров,147,137
2,Сидорова,217,120


#### df.select_dtypes() — выбор колонок по типу данных

Незаменимо при автоматической обработке: например, применить агрегацию ко всем числовым колонкам.


In [175]:
df_mixed = pd.DataFrame({
    "client_id": [1, 2, 3],
    "name":      ["А", "Б", "В"],
    "revenue":   [100.5, 200.7, 175.3],
    "date":      pd.date_range("2026-01-01", periods=3),
    "is_active": [True, False, True],
})

print("Все числовые:")
print(df_mixed.select_dtypes(include="number").columns.tolist())

print("\nВсе, кроме чисел:")
print(df_mixed.select_dtypes(exclude="number").columns.tolist())

# Конкретные типы
df_mixed.select_dtypes(include=["float64", "bool"])

Все числовые:
['client_id', 'revenue']

Все, кроме чисел:
['name', 'date', 'is_active']


,revenue,is_active
0,100.5,True
1,200.7,False
2,175.3,True


#### Массовое переименование через паттерн

Часто нужно очистить названия колонок (привести к snake_case, убрать пробелы).


In [176]:
# Пример: имена колонок с лишними пробелами и заглавными буквами
df_dirty = pd.DataFrame({
    "  Client Name  ": ["А", "Б"],
    "Revenue (RUB)":   [100, 200],
    "ORDER COUNT":     [2, 5],
})

# Нормализация
df_dirty.columns = (
    df_dirty.columns
    .str.strip()                    # убрать пробелы по краям
    .str.lower()                    # в нижний регистр
    .str.replace(r"\s+", "_", regex=True)   # пробелы -> _
    .str.replace(r"[^\w]", "", regex=True)  # убрать спецсимволы
)
df_dirty

,client_name,revenue_rub,order_count
0,А,100,2
1,Б,200,5


---
## 9. Выбор строк и фильтрация

### 9.1 iloc — выбор по числовой позиции

`iloc` работает с **целочисленными позициями** (как обычные Python-индексы). Позиции начинаются с 0, правая граница среза **не включается**.

Синтаксис: `df.iloc[строки, колонки]`


In [177]:
df.iloc[0]              # Первая строка

date           2026-01-01 00:00:00
manager                   Сидорова
region                      Москва
revenue                     431236
cost                        728843
margin                     -297607
margin_rate                -0.6901
orders                          18
Name: 0, dtype: object

In [178]:
df.iloc[:5]             # Первые 5 строк

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2
3,2026-01-04,Сидорова,СПб,589492,306508,282984,0.4800,10
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4


In [179]:
df.iloc[5:10]           # Строки 5–9

,date,manager,region,revenue,cost,margin,margin_rate,orders
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
6,2026-01-07,Сидорова,Москва,449457,156530,292927,0.6517,16
7,2026-01-08,Петров,Казань,765987,654365,111622,0.1457,15
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,1388507,510337,878170,0.6325,14


In [180]:
df.iloc[-3:]            # Последние 3 строки

,date,manager,region,revenue,cost,margin,margin_rate,orders
17,2026-01-18,Петров,Казань,748531,221829,526702,0.7036,15
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1


In [181]:
df.iloc[0, 2]           # Строка 0, колонка 2

'Москва'

In [182]:
df.iloc[[0,3,7], :3]    # Строки 0,3,7 — первые 3 колонки

,date,manager,region
0,2026-01-01,Сидорова,Москва
3,2026-01-04,Сидорова,СПб
7,2026-01-08,Петров,Казань


### 9.2 loc — выбор по меткам

`loc` работает с **метками** (именами индексов и колонок). Важное отличие от `iloc`: правая граница среза **включается**.

`loc` — основной инструмент для **изменения** значений: `df.loc[условие, "col"] = value`.


In [183]:
df.loc[0, "revenue"]                    # ячейка

np.int64(431236)

In [184]:
df.loc[0:4, ["manager","revenue"]]      # строки 0–4 включительно, две колонки

,manager,revenue
0,Сидорова,431236
1,Иванов,628178
2,Сидорова,665894
3,Сидорова,589492
4,Иванов,1405416


In [185]:
# Изменение значений через loc (правильный способ!)
df_copy = df.copy()
df_copy.loc[df_copy["revenue"] > 1_500_000, "segment"] = "VIP"
df_copy[["manager","revenue","segment"]].dropna().head()

,manager,revenue,segment
13,Сидорова,1880488,VIP
14,Петров,1565689,VIP
16,Петров,1925665,VIP


### 9.3 Булева фильтрация

Самый частый способ отбора строк. `df["col"] > value` возвращает Series из `True`/`False`. При передаче в `df[...]` остаются только строки с `True`.

**Правила для нескольких условий:**
- вместо Python-оператора `and` используй `&`
- вместо `or` используй `|`
- вместо `not` используй `~`
- каждое условие **обязательно** в круглых скобках

Причина: оператор `&` имеет более высокий приоритет, чем `>` и `<`. Без скобок `df["a"] > 0 & df["b"] < 10` интерпретируется как `df["a"] > (0 & df["b"]) < 10` — что совсем не то, что нужно.


In [186]:
df[df["revenue"] > 1_000_000]

,date,manager,region,revenue,cost,margin,margin_rate,orders
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,1388507,510337,878170,0.6325,14
10,2026-01-11,Сидорова,СПб,1388205,774839,613366,0.4418,8
11,2026-01-12,Сидорова,Казань,1353617,855889,497728,0.3677,16
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1


In [187]:
# AND — оба условия
df[(df["revenue"] > 1_000_000) & (df["margin"] > 0)]

,date,manager,region,revenue,cost,margin,margin_rate,orders
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,1388507,510337,878170,0.6325,14
10,2026-01-11,Сидорова,СПб,1388205,774839,613366,0.4418,8
11,2026-01-12,Сидорова,Казань,1353617,855889,497728,0.3677,16
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1


In [188]:
# OR — хотя бы одно
df[(df["region"] == "Москва") | (df["revenue"] > 1_500_000)]

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
6,2026-01-07,Сидорова,Москва,449457,156530,292927,0.6517,16
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13


In [189]:
# NOT — инверсия
df[~(df["region"] == "Казань")]

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2
3,2026-01-04,Сидорова,СПб,589492,306508,282984,0.4800,10
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
6,2026-01-07,Сидорова,Москва,449457,156530,292927,0.6517,16
10,2026-01-11,Сидорова,СПб,1388205,774839,613366,0.4418,8
12,2026-01-13,Иванов,СПб,798361,509773,288588,0.3615,13
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18


In [190]:
# isin — значение из списка
df[df["region"].isin(["Москва", "СПб"])]

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2
3,2026-01-04,Сидорова,СПб,589492,306508,282984,0.4800,10
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
6,2026-01-07,Сидорова,Москва,449457,156530,292927,0.6517,16
10,2026-01-11,Сидорова,СПб,1388205,774839,613366,0.4418,8
12,2026-01-13,Иванов,СПб,798361,509773,288588,0.3615,13
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18


In [191]:
# between — числовой диапазон (оба края включаются)
df[df["revenue"].between(500_000, 1_200_000)]

,date,manager,region,revenue,cost,margin,margin_rate,orders
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2
3,2026-01-04,Сидорова,СПб,589492,306508,282984,0.4800,10
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
7,2026-01-08,Петров,Казань,765987,654365,111622,0.1457,15
12,2026-01-13,Иванов,СПб,798361,509773,288588,0.3615,13
17,2026-01-18,Петров,Казань,748531,221829,526702,0.7036,15
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1


### 9.4 query() — читаемый строковый синтаксис

`query()` позволяет записать условия строкой. Код чище, особенно при сложных условиях. Ссылка на Python-переменную — через `@`.


In [192]:
df.query("revenue > 1_000_000")

,date,manager,region,revenue,cost,margin,margin_rate,orders
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,1388507,510337,878170,0.6325,14
10,2026-01-11,Сидорова,СПб,1388205,774839,613366,0.4418,8
11,2026-01-12,Сидорова,Казань,1353617,855889,497728,0.3677,16
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1


In [193]:
df.query("region == 'Москва' and revenue > 500_000")

,date,manager,region,revenue,cost,margin,margin_rate,orders
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13


In [194]:
threshold = 800_000
df.query("revenue > @threshold and margin > 0")

,date,manager,region,revenue,cost,margin,margin_rate,orders
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,1388507,510337,878170,0.6325,14
10,2026-01-11,Сидорова,СПб,1388205,774839,613366,0.4418,8
11,2026-01-12,Сидорова,Казань,1353617,855889,497728,0.3677,16
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
18,2026-01-19,Петров,Москва,1400571,321836,1078735,0.7702,13
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1


### 9.5 mask() и where() — условная замена значений

`where(cond)` оставляет значения, где условие `True`, остальные заменяет на `NaN` (или другое значение).
`mask(cond)` — наоборот: заменяет там, где условие `True`.

Это полезно для **очистки данных без удаления строк** и для обработки выбросов.


In [195]:
s = pd.Series([10, 20, 30, 40, 50])

# where — оставить только > 25, остальное -> NaN
print(s.where(s > 25))

# where с заменой
print(s.where(s > 25, 0))           # < 25 -> 0

# mask — обратная логика: заменить там, где True
print(s.mask(s > 25))               # > 25 -> NaN
print(s.mask(s > 25, "high"))       # > 25 -> "high"

0     NaN
1     NaN
2    30.0
3    40.0
4    50.0
dtype: float64
0     0
1     0
2    30
3    40
4    50
dtype: int64
0    10.0
1    20.0
2     NaN
3     NaN
4     NaN
dtype: float64
0      10
1      20
2    high
3    high
4    high
dtype: object


In [196]:
# Практика: ограничение выбросов (capping)
np.random.seed(0)
df = pd.DataFrame({
    "revenue": np.random.randint(100, 10_000, 10).tolist() + [50_000, 100_000]  # 2 выброса
})
df = df.head(12)

# Ограничить выручку 95-м перцентилем
cap = df["revenue"].quantile(0.95)
df["revenue_capped"] = df["revenue"].mask(df["revenue"] > cap, cap)
df

/var/folders/ls/d6mbmny12hb59h29c1bx9rsm0000gn/T/ipykernel_3594/3062646992.py:10: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["revenue_capped"] = df["revenue"].mask(df["revenue"] > cap, cap)


,revenue,revenue_capped
0,2832,2832
1,9945,9945
2,3364,3364
3,4959,4959
4,9325,9325
5,7991,7991
6,4473,4473
7,5974,5974
8,6844,6844
9,3568,3568


### 9.6 isin / between — расширенные паттерны

`isin()` принимает список, **другую Series**, **dict** или **DataFrame**. Это позволяет делать сложные фильтры компактно.


In [197]:
# isin с другой колонкой / Series
np.random.seed(0)
df_ord = pd.DataFrame({
    "order_id":  [1, 2, 3, 4, 5],
    "client_id": [101, 102, 103, 101, 999],
})
allowed_clients = pd.Series([101, 102, 103])

# Заказы только от разрешённых клиентов
df_ord[df_ord["client_id"].isin(allowed_clients)]

,order_id,client_id
0,1,101
1,2,102
2,3,103
3,4,101


In [198]:
# Антипаттерн: пересечение — isin + ~
all_clients = pd.Series([101, 102, 103, 104, 105])
in_db       = pd.Series([101, 102, 999])

# Клиенты, которых нет в БД
missing = all_clients[~all_clients.isin(in_db)]
print(missing.tolist())

[103, 104, 105]


---
## 10. Индексы

**Индекс** — метки строк DataFrame. По умолчанию pandas создаёт числовой `RangeIndex(0, n)`. Но индексом может быть любой столбец: строки, даты, составные ключи.

### Зачем нужен кастомный индекс?
- Ускоряет поиск строки через `loc`
- Обязателен для временных рядов (`DatetimeIndex` + `resample`)
- Нужен при выравнивании нескольких Series при арифметике

### Практическая рекомендация

В большинстве аналитических задач лучше держать **числовой индекс** и сбрасывать его через `reset_index()` после операций groupby/merge. Кастомный индекс полезен именно для временных рядов и словарных lookups.


In [199]:
df_idx = df.set_index("date")
df_idx.head(3)

KeyError: "None of ['date'] are in the columns"

In [ ]:
# Сбросить индекс -> вернуть в колонку
df_reset = df_idx.reset_index()
df_reset.head(3)

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2


In [ ]:
# Сбросить и выбросить
df_reset2 = df_idx.reset_index(drop=True)
df_reset2.head(3)

,manager,region,revenue,cost,margin,margin_rate,orders
0,Сидорова,Москва,431236,728843,-297607,-0.6901,18
1,Иванов,Москва,628178,960790,-332612,-0.5295,12
2,Сидорова,СПб,665894,622843,43051,0.0647,2


In [ ]:
print("Индекс уникален:", df.index.is_unique)
df_sorted_idx = df.sort_index()

Индекс уникален: True


---
## 11. Типы данных

Правильные типы данных — фундамент качественного анализа. Неправильные типы приводят к:
- ошибочным результатам вычислений (строки вместо чисел)
- избыточному потреблению памяти
- медленным группировкам и фильтрам

### Основные типы pandas

| Тип pandas | Описание | Пример |
|-----------|----------|--------|
| `int64`, `int32`, `int16` | Целые числа | id, количество |
| `float64`, `float32` | Числа с плавающей точкой | цена, выручка |
| `object` | Строки (устаревший вариант) | имена, категории |
| `string` | Строки (pandas StringDtype) | имена — современный вариант |
| `bool` | Булев тип | флаги |
| `category` | Категориальный | регион, статус, сегмент |
| `datetime64[ns]` | Дата и время | дата транзакции |

### Правило выбора типа
- Числа → `int` или `float` наименьшего подходящего размера
- Строки с малым кол-вом уникальных значений → **`category`**
- Даты → `datetime64` через `pd.to_datetime()`


In [ ]:
df.dtypes

date           datetime64[ns]
manager                object
region                 object
revenue                 int64
cost                    int64
margin                  int64
margin_rate           float64
orders                  int64
dtype: object

In [ ]:
df_types = df.copy()
df_types["orders"]  = df_types["orders"].astype("int32")      # экономит память
df_types["region"]  = df_types["region"].astype("category")   # строки -> категории
df_types["manager"] = df_types["manager"].astype("string")    # pandas StringDtype

before = df.memory_usage(deep=True).sum()
after  = df_types.memory_usage(deep=True).sum()
print(f"До:    {before:,} байт")
print(f"После: {after:,} байт  (экономия {(1-after/before)*100:.0f}%)")

До:    5,368 байт
После: 3,657 байт  (экономия 32%)


In [ ]:
# pd.to_numeric — безопасное преобразование строк в числа
# errors="coerce": нечисловое -> NaN (не падает с ошибкой)
sample = pd.Series(["1000", "2500", "нет данных", "3000", ""])
pd.to_numeric(sample, errors="coerce")

0    1000.0
1    2500.0
2       NaN
3    3000.0
4       NaN
dtype: float64

In [ ]:
# pd.to_datetime — преобразование строк в даты
dates = pd.Series(["2026-01-15", "2026-02-28", "не дата", "2026-04-01"])
pd.to_datetime(dates, errors="coerce")

0   2026-01-15
1   2026-02-28
2          NaT
3   2026-04-01
dtype: datetime64[ns]

### Почему category экономит память

`category` хранит строки не как отдельные объекты, а как **массив целых чисел + словарь**:

```
object:   ["Москва","СПб","Казань","Москва","Москва"]  → 5 строк × ~50 байт = 250 байт
category: [1, 2, 0, 1, 1] + {0:"Казань",1:"Москва",2:"СПб"} → ~25 байт
```

На миллионе строк с 10 уникальными значениями — экономия **в десятки раз**.


In [ ]:
s_obj = df["region"].astype("object")
s_cat = df["region"].astype("category")
print(f"object:   {s_obj.memory_usage(deep=True):,} байт")
print(f"category: {s_cat.memory_usage(deep=True):,} байт")

object:   2,196 байт
category: 565 байт


### 11.5 Nullable types (Int64, Float64, boolean) — современные типы с поддержкой пропусков

В классическом NumPy/pandas есть проблема: **целочисленные колонки не могут содержать пропуски**. При появлении `NaN` колонка автоматически конвертируется в `float64`. Это неудобно при работе с ID и счётчиками.

Решение — **nullable types** (с большой буквы: `Int64`, `Float64`, `boolean`, `string`). Они нативно поддерживают `pd.NA` — универсальное обозначение пропуска.


In [ ]:
# Классический int — не может быть NaN, конвертируется в float
s_classic = pd.Series([1, 2, None, 4])
print("Классический:")
print(s_classic)
print("dtype:", s_classic.dtype)   # float64 (был int!)

In [ ]:
# Nullable Int64 — сохраняет целые числа + поддерживает пропуски
s_nullable = pd.Series([1, 2, None, 4], dtype="Int64")
print("Nullable:")
print(s_nullable)
print("dtype:", s_nullable.dtype)   # Int64

# Арифметика работает обычным образом
print(s_nullable * 10)

In [ ]:
# nullable boolean — поддерживает True / False / pd.NA
s_bool = pd.Series([True, False, None, True], dtype="boolean")
print(s_bool)

# Поведение pd.NA в логических операциях аккуратное
print(s_bool & True)
print(s_bool | False)

---
## 12. Пропуски (Missing Values)

Пропуски — нормальная часть реальных данных. Они возникают из-за:
- ошибок ввода или выгрузки
- отсутствующих связей при JOIN
- незаполненных полей в CRM/ERP

В pandas пропуск обозначается `NaN` для чисел и `NaT` для дат.

### Стратегии работы с пропусками

| Стратегия | Когда применять |
|-----------|----------------|
| `dropna()` | пропусков мало, строки некритичны |
| `fillna(0)` | пропуск = нулевое значение (нет продаж = 0) |
| `fillna(mean)` | числовые данные без выбросов |
| `fillna(median)` | числовые данные с выбросами |
| `fillna("unknown")` | строки, категория неизвестна |
| `ffill() / bfill()` | временные ряды |
| `groupby.transform` | заполнить по среднему группы |

> **Важно:** не заполняйте пропуски бездумно. Иногда `NaN` несёт смысловую нагрузку — например, «клиент не совершал покупок». Удаление или замена меняет распределение.


### 12.0 Природа пропусков: MCAR / MAR / MNAR

В статистике пропуски классифицируют по причине их возникновения. Это влияет на **выбор стратегии** заполнения.

| Тип | Расшифровка | Пример |
|-----|-------------|--------|
| **MCAR** | Missing Completely At Random | случайный сбой системы выгрузки |
| **MAR** | Missing At Random | пропуск зависит от других колонок (например, мужчины не указывают возраст чаще женщин) |
| **MNAR** | Missing Not At Random | пропуск зависит от самого значения (богатые клиенты не указывают доход) |

### Чем опасны пропуски

- **MCAR** → пропуски можно удалить или заполнить медианой/средним
- **MAR** → нужно заполнять с учётом группировки (`groupby + transform`)
- **MNAR** → серьёзная проблема: простое заполнение исказит данные

> **Прежде чем заполнять, ВСЕГДА** изучите распределение пропусков. Если их 50% в одной группе и 1% в другой — это сигнал.


In [ ]:
df_null = df.copy()
df_null.loc[[2, 5, 9, 14], "revenue"] = np.nan
df_null.loc[[1, 7, 11],    "manager"] = np.nan

print("Пропуски по колонкам:")
print(df_null.isna().sum())

Пропуски по колонкам:
date           0
manager        3
region         0
revenue        4
cost           0
margin         0
margin_rate    0
orders         0
dtype: int64


In [ ]:
# Доля пропусков (%)
(df_null.isna().mean() * 100).round(2)

date            0.0
manager        15.0
region          0.0
revenue        20.0
cost            0.0
margin          0.0
margin_rate     0.0
orders          0.0
dtype: float64

In [ ]:
df_null[df_null["revenue"].isna()]     # строки с пропуском

,date,manager,region,revenue,cost,margin,margin_rate,orders
2,2026-01-03,Сидорова,СПб,NaN,622843,43051,0.0647,2
5,2026-01-06,Иванов,Москва,NaN,946942,-373688,-0.6519,14
9,2026-01-10,Сидорова,Казань,NaN,510337,878170,0.6325,14
14,2026-01-15,Петров,Казань,NaN,814469,751220,0.4798,15


In [ ]:
df_null[df_null["revenue"].notna()]    # строки без пропуска

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236.0,728843,-297607,-0.6901,18
1,2026-01-02,NaN,Москва,628178.0,960790,-332612,-0.5295,12
3,2026-01-04,Сидорова,СПб,589492.0,306508,282984,0.4800,10
4,2026-01-05,Иванов,Москва,1405416.0,853591,551825,0.3926,4
6,2026-01-07,Сидорова,Москва,449457.0,156530,292927,0.6517,16
7,2026-01-08,NaN,Казань,765987.0,654365,111622,0.1457,15
8,2026-01-09,Сидорова,Казань,1419512.0,872352,547160,0.3855,8
10,2026-01-11,Сидорова,СПб,1388205.0,774839,613366,0.4418,8
11,2026-01-12,NaN,Казань,1353617.0,855889,497728,0.3677,16
12,2026-01-13,Иванов,СПб,798361.0,509773,288588,0.3615,13


In [ ]:
# Удаление
df_null.dropna()                          # любой пропуск в строке
df_null.dropna(subset=["revenue"])        # пропуск в конкретной колонке
df_null.dropna(thresh=5)                  # оставить строки с >= 5 непустых

,date,manager,region,revenue,cost,margin,margin_rate,orders
0,2026-01-01,Сидорова,Москва,431236.0,728843,-297607,-0.6901,18
1,2026-01-02,NaN,Москва,628178.0,960790,-332612,-0.5295,12
2,2026-01-03,Сидорова,СПб,NaN,622843,43051,0.0647,2
3,2026-01-04,Сидорова,СПб,589492.0,306508,282984,0.4800,10
4,2026-01-05,Иванов,Москва,1405416.0,853591,551825,0.3926,4
5,2026-01-06,Иванов,Москва,NaN,946942,-373688,-0.6519,14
6,2026-01-07,Сидорова,Москва,449457.0,156530,292927,0.6517,16
7,2026-01-08,NaN,Казань,765987.0,654365,111622,0.1457,15
8,2026-01-09,Сидорова,Казань,1419512.0,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,NaN,510337,878170,0.6325,14


In [ ]:
df_f = df_null.copy()
df_f["revenue"] = df_f["revenue"].fillna(df_f["revenue"].median())
df_f["manager"] = df_f["manager"].fillna("Не назначен")
# Временные ряды:
# df_f["revenue"] = df_f["revenue"].ffill()
# df_f["revenue"] = df_f["revenue"].bfill()
df_f.isna().sum()

date           0
manager        0
region         0
revenue        0
cost           0
margin         0
margin_rate    0
orders         0
dtype: int64

In [ ]:
# Заполнение средним по группе — мощный приём
df_null["revenue"] = df_null["revenue"].fillna(
    df_null.groupby("region")["revenue"].transform("mean")
)
df_null.isna().sum()

date           0
manager        3
region         0
revenue        0
cost           0
margin         0
margin_rate    0
orders         0
dtype: int64

### 12.7 Расширенные паттерны работы с пропусками

#### interpolate() — интерполяция (особенно для временных рядов)

Заполняет пропуски значениями, **рассчитанными по соседним точкам**. Это особенно подходит для гладких рядов (температура, курс валют, медленно меняющиеся метрики).


In [ ]:
# Временной ряд с пропусками
np.random.seed(0)
df_temp = pd.DataFrame({
    "date":   pd.date_range("2026-01-01", periods=10, freq="D"),
    "temp_c": [-5.0, np.nan, np.nan, -2.0, 0.0, np.nan, 3.0, 5.0, np.nan, 7.0],
})

# Линейная интерполяция
df_temp["temp_linear"] = df_temp["temp_c"].interpolate(method="linear")

# Заполнение предыдущим значением (для сравнения)
df_temp["temp_ffill"]  = df_temp["temp_c"].ffill()

df_temp

#### Подсчёт пропусков по строкам, а не по колонкам


In [ ]:
# Сколько пропусков в каждой строке
df_null = pd.DataFrame({
    "a": [1, None, 3, None],
    "b": [None, 2, 3, None],
    "c": [1, 2, None, None],
})
df_null["nulls_in_row"] = df_null.isna().sum(axis=1)
df_null

# Оставить только строки, где не более 1 пропуска
# df_null[df_null["nulls_in_row"] <= 1]

#### Преобразование «пустых» значений в NaN

В реальных данных пропуски часто маскируются под пустые строки `""`, `"-"`, `"N/A"`, `"нет"`, `0` (когда 0 — не настоящее значение).


In [ ]:
df_dirty = pd.DataFrame({
    "client": ["Alpha", "Beta", "", "-", "Gamma"],
    "amount": ["1000", "N/A", "", "нет", "500"],
})

# replace -> NaN
df_dirty = df_dirty.replace(
    to_replace=["", "-", "N/A", "нет"],
    value=np.nan
)

# Теперь можно нормально работать
df_dirty["amount"] = pd.to_numeric(df_dirty["amount"], errors="coerce")
df_dirty

---
## 13. Дубликаты

Дубликаты — строки, полностью или частично повторяющие другие. Они появляются из-за двойной загрузки данных, ошибок при JOIN, дублей в источнике.

### Типы дублей
- **Полный дубль** — все колонки совпадают
- **Частичный дубль** — совпадают только ключевые поля (например, `client_id` и `date`)

`duplicated()` возвращает `True/False`. По умолчанию `keep="first"` — первое вхождение считается оригиналом.


In [ ]:
df_dup = pd.concat([df, df.iloc[:3]], ignore_index=True)
print(f"Дублей: {df_dup.duplicated().sum()} из {len(df_dup)} строк")

Дублей: 3 из 23 строк


In [ ]:
# Посмотреть все копии (keep=False помечает ВСЕ, включая первую)
df_dup[df_dup.duplicated(keep=False)].sort_values("manager")

,date,manager,region,revenue,cost,margin,margin_rate,orders
1,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
21,2026-01-02,Иванов,Москва,628178,960790,-332612,-0.5295,12
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
2,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2
20,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
22,2026-01-03,Сидорова,СПб,665894,622843,43051,0.0647,2


In [ ]:
df_clean = df_dup.drop_duplicates()
print(f"После очистки: {len(df_clean)} строк")

# Оставить последнюю запись
df_clean_last = df_dup.drop_duplicates(keep="last")

# По конкретным колонкам
df_one_per_mgr = df_dup.drop_duplicates(subset=["manager"])
print(f"Уникальных менеджеров: {len(df_one_per_mgr)}")

После очистки: 20 строк
Уникальных менеджеров: 3


---
## 14. Сортировка

`sort_values()` сортирует DataFrame по значениям колонок. Сортировка не меняет исходный DataFrame — присваивайте результат.

Параметры:
- `ascending=True` — по возрастанию (умолчание)
- `na_position="last"` — куда помещать `NaN` (умолчание: в конец)


In [ ]:
df.sort_values("revenue").head()

,date,manager,region,revenue,cost,margin,margin_rate,orders
15,2026-01-16,Иванов,СПб,168148,391097,-222949,-1.3259,13
0,2026-01-01,Сидорова,Москва,431236,728843,-297607,-0.6901,18
6,2026-01-07,Сидорова,Москва,449457,156530,292927,0.6517,16
5,2026-01-06,Иванов,Москва,573254,946942,-373688,-0.6519,14
3,2026-01-04,Сидорова,СПб,589492,306508,282984,0.4800,10


In [ ]:
df.sort_values("revenue", ascending=False).head()

,date,manager,region,revenue,cost,margin,margin_rate,orders
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
13,2026-01-14,Сидорова,СПб,1880488,258261,1622227,0.8627,18
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
4,2026-01-05,Иванов,Москва,1405416,853591,551825,0.3926,4


In [ ]:
# По нескольким: сначала регион (А→Я), внутри — выручка (↓)
df.sort_values(["region","revenue"], ascending=[True, False]).head(8)

,date,manager,region,revenue,cost,margin,margin_rate,orders
16,2026-01-17,Петров,Казань,1925665,365139,1560526,0.8104,9
14,2026-01-15,Петров,Казань,1565689,814469,751220,0.4798,15
8,2026-01-09,Сидорова,Казань,1419512,872352,547160,0.3855,8
9,2026-01-10,Сидорова,Казань,1388507,510337,878170,0.6325,14
11,2026-01-12,Сидорова,Казань,1353617,855889,497728,0.3677,16
19,2026-01-20,Петров,Казань,1006606,488974,517632,0.5142,1
7,2026-01-08,Петров,Казань,765987,654365,111622,0.1457,15
17,2026-01-18,Петров,Казань,748531,221829,526702,0.7036,15


In [ ]:
df.nlargest(5, "revenue")[["manager","region","revenue"]]

,manager,region,revenue
16,Петров,Казань,1925665
13,Сидорова,СПб,1880488
14,Петров,Казань,1565689
8,Сидорова,Казань,1419512
4,Иванов,Москва,1405416


In [ ]:
df.nsmallest(5, "margin")[["manager","region","margin"]]
# делает две операции подряд:
#     находит 5 строк с самым большим значением в колонке revenue
#     показывает только колонки manager, region, revenue

,manager,region,margin
5,Иванов,Москва,-373688
1,Иванов,Москва,-332612
0,Сидорова,Москва,-297607
15,Иванов,СПб,-222949
2,Сидорова,СПб,43051


---
## 15. Базовые расчёты

### Векторные операции — основа производительности

Вместо циклов `for` pandas использует **векторные операции** — математические действия применяются к целой колонке одновременно. Под капотом это оптимизированные NumPy-операции на C.

```python
# ❌ Медленно — цикл на Python (тысячи раз медленнее)
for i in range(len(df)):
    df.at[i, "margin"] = df.at[i, "revenue"] - df.at[i, "cost"]

# ✅ Быстро — векторная операция
df["margin"] = df["revenue"] - df["cost"]
```


### 15.0 Векторизация под капотом

Когда вы пишете `df["a"] + df["b"]`, pandas:
1. Извлекает NumPy-массивы из обеих колонок
2. Передаёт их в **скомпилированный код C/Fortran**, который складывает поэлементно
3. Заворачивает результат обратно в Series

Цикл `for` на Python проигрывает в 100–1000 раз именно потому, что каждая итерация — это вызов интерпретатора. Векторная операция — один вызов C-функции на весь массив.

### Какие операции векторизованы

- **Арифметика**: `+`, `-`, `*`, `/`, `**`, `%`
- **Сравнения**: `>`, `<`, `==`, `!=`, `>=`, `<=`
- **Логика**: `&`, `|`, `~`
- **Функции NumPy**: `np.log`, `np.sqrt`, `np.exp`, `np.abs`...
- **Большинство методов pandas**: `.sum()`, `.mean()`, `.cumsum()`, `.diff()`, и т.д.

Эти операции — **бесплатные** с точки зрения времени Python.


In [ ]:
df["margin"]      = df["revenue"] - df["cost"]
df["margin_rate"] = (df["margin"] / df["revenue"]).round(4)

df[["manager","revenue","cost","margin","margin_rate"]].head()

,manager,revenue,cost,margin,margin_rate
0,Сидорова,431236,728843,-297607,-0.6901
1,Иванов,628178,960790,-332612,-0.5295
2,Сидорова,665894,622843,43051,0.0647
3,Сидорова,589492,306508,282984,0.4800
4,Иванов,1405416,853591,551825,0.3926


In [ ]:
r = df["revenue"]
print(f"Сумма:       {r.sum():>15,.0f}")
print(f"Среднее:     {r.mean():>15,.0f}")
print(f"Медиана:     {r.median():>15,.0f}")
print(f"Ст. откл.:   {r.std():>15,.0f}")
print(f"Мин:         {r.min():>15,.0f}")
print(f"Макс:        {r.max():>15,.0f}")
print(f"Квантиль 90: {r.quantile(0.9):>15,.0f}")

Сумма:            20,552,814
Среднее:           1,027,641
Медиана:             902,484
Ст. откл.:           510,011
Мин:                 168,148
Макс:              1,925,665
Квантиль 90:       1,597,169


In [ ]:
idx_max = df["revenue"].idxmax()
# Эта строка находит индекс строки, где в колонке revenue находится максимальное значение.
print("Лучшая сделка:")
print(df.loc[idx_max, ["manager","region","revenue"]])

Лучшая сделка:
manager     Петров
region      Казань
revenue    1925665
Name: 16, dtype: object


In [ ]:
df["margin_rate_pct"] = (df["margin_rate"] * 100).round(1)
df["abs_margin"]     = df["margin"].abs()

# Нарастающий итог
df_s = df.sort_values("date")
df_s["revenue_cumsum"] = df_s["revenue"].cumsum()
df_s[["date","revenue","revenue_cumsum"]].head(6)

KeyError: 'margin_rate'

### 15.5 Накопительные функции (cumulative)

Помимо `cumsum`, есть полный набор «накопительных» операций. Они обходят Series слева направо и возвращают результат той же длины.

| Метод | Назначение |
|-------|-----------|
| `cumsum()` | накопительная сумма |
| `cumprod()` | накопительное произведение |
| `cummax()` | накопительный максимум (бегущий) |
| `cummin()` | накопительный минимум |
| `cumcount()` | порядковый номер внутри группы |


In [ ]:
np.random.seed(0)
df_c = pd.DataFrame({
    "month":   ["Янв","Фев","Мар","Апр","Май","Июн"],
    "revenue": [100, 120, 90, 150, 130, 180],
})

df_c["cumsum"]  = df_c["revenue"].cumsum()    # нарастающий итог
df_c["cummax"]  = df_c["revenue"].cummax()    # пиковое значение к моменту
df_c["cummin"]  = df_c["revenue"].cummin()    # минимум к моменту

# Накопительный темп роста
df_c["base"] = df_c["revenue"].iloc[0]
df_c["growth_vs_first"] = (df_c["revenue"] / df_c["base"] - 1).round(2)
df_c.drop(columns="base")

### 15.6 diff() — разница с предыдущим значением

`diff(n)` возвращает разницу между текущим и значением на `n` строк выше. Это базовый инструмент для расчёта изменений.


In [ ]:
df_d = pd.DataFrame({
    "date":    pd.date_range("2026-01-01", periods=6, freq="D"),
    "revenue": [100, 120, 90, 150, 130, 180],
})

df_d["diff_1d"]  = df_d["revenue"].diff(1)        # изменение к предыдущему дню
df_d["diff_2d"]  = df_d["revenue"].diff(2)        # к 2 дням назад
df_d["growth_%"] = (df_d["revenue"].pct_change() * 100).round(1)

df_d

### 15.7 agg() — несколько агрегатов сразу

Метод `.agg()` (синоним `.aggregate()`) принимает список функций или словарь и возвращает все агрегаты разом. Удобно для отчётов.


In [ ]:
# Несколько агрегатов на одну колонку
df_d["revenue"].agg(["sum", "mean", "median", "std", "min", "max"]).round(1)

In [ ]:
# Разные функции к разным колонкам — через словарь
df_m = pd.DataFrame({
    "revenue": [100, 200, 300],
    "orders":  [2, 5, 4],
    "manager": ["А", "Б", "А"],
})
df_m.agg({"revenue": ["sum", "mean"], "orders": "sum"})

---
## 16. Условные колонки

| Инструмент | Условий | Скорость |
|-----------|---------|---------|
| `np.where` | 1 (if/else) | ⚡ |
| `np.select` | много | ⚡ |
| `pd.cut` | диапазоны числовой колонки | ⚡ |
| `pd.qcut` | квантили | ⚡ |
| `df.apply(axis=1)` | сложная логика | 🐌 |

### 16.1 np.where — одно условие


In [ ]:
df["is_profitable"] = np.where(df["margin"] > 0, "прибыльная", "убыточная")
df[["manager","revenue","margin","is_profitable"]].head()

,manager,revenue,margin,is_profitable
0,Сидорова,431236,-297607,убыточная
1,Иванов,628178,-332612,убыточная
2,Сидорова,665894,43051,прибыльная
3,Сидорова,589492,282984,прибыльная
4,Иванов,1405416,551825,прибыльная


### 16.2 np.select — много условий

Условия проверяются **сверху вниз**, применяется первое подходящее.

In [ ]:
conditions = [
    df["revenue"] >= 1_500_000,
    df["revenue"] >= 800_000,
    df["revenue"] >= 400_000,
]
choices = ["VIP", "крупный", "средний"]
df["client_size"] = np.select(conditions, choices, default="малый")
df["client_size"].value_counts()

client_size
средний    9
крупный    7
VIP        3
малый      1
Name: count, dtype: int64

### 16.3 pd.cut — разбивка на интервалы

Удобно, когда у вас есть **бизнес-логика** разбивки: «до 500К — малый, 500К–1М — средний».

In [ ]:
df["revenue_band"] = pd.cut(
    df["revenue"],
    bins   = [0, 500_000, 1_000_000, 1_500_000, np.inf],
    labels = ["< 500K", "500K–1M", "1M–1.5M", "> 1.5M"],
    right  = True   # правая граница включается
)
df["revenue_band"].value_counts().sort_index()

revenue_band
< 500K     3
500K–1M    7
1M–1.5M    7
> 1.5M     3
Name: count, dtype: int64

### 16.4 pd.qcut — разбивка по квантилям

Делит на **равные по количеству** группы. Полезно для ранжирования без фиксированных порогов.

In [ ]:
df["revenue_quartile"] = pd.qcut(
    df["revenue"], q=4, labels=["Q1","Q2","Q3","Q4"]
)
df["revenue_quartile"].value_counts()

revenue_quartile
Q1    5
Q2    5
Q3    5
Q4    5
Name: count, dtype: int64

---
## 17. apply и map

### 17.1 map() — замена значений по словарю

`Series.map(dict)` заменяет каждое значение согласно словарю. Это быстрая операция. Если значение не найдено — ставится `NaN`.


In [ ]:
region_en = {"Москва": "Moscow", "СПб": "Saint Petersburg", "Казань": "Kazan"}
df["region_en"] = df["region"].map(region_en)
df[["region","region_en"]].value_counts()

region  region_en       
Казань  Kazan               8
Москва  Moscow              6
СПб     Saint Petersburg    6
Name: count, dtype: int64

### 17.2 apply() — функция к Series

`Series.apply(func)` применяет функцию к каждому элементу. Медленнее векторных операций, но гибче.

In [ ]:
df["manager_abbr"] = df["manager"].apply(lambda x: x[:3].upper())
# Эквивалент — быстрее:
# df["manager_abbr"] = df["manager"].str[:3].str.upper()
df[["manager","manager_abbr"]].drop_duplicates()

,manager,manager_abbr
0,Сидорова,СИД
1,Иванов,ИВА
7,Петров,ПЕТ


### 17.3 apply() по строкам (axis=1)

`DataFrame.apply(func, axis=1)` передаёт каждую **строку** как Series. Мощно, но работает на Python без векторизации — **в десятки раз медленнее** векторных операций.

**Правило:** сначала пробуй `np.where`, `np.select`, арифметику. `apply(axis=1)` — только когда ничего другого не подходит.


In [ ]:
def classify_deal(row):
    if row["margin"] < 0:
        return "убыточная"
    elif row["margin_rate"] < 0.1:
        return "низкомаржинальная"
    elif row["revenue"] > 1_000_000 and row["margin_rate"] > 0.3:
        return "флагман"
    else:
        return "стандартная"

df["deal_class"] = df.apply(classify_deal, axis=1)
df["deal_class"].value_counts()

deal_class
флагман              10
стандартная           5
убыточная             4
низкомаржинальная     1
Name: count, dtype: int64

### 17.4 Иерархия скоростей: какой инструмент выбрать

При вычислениях на колонках производительность различается **в сотни раз**:

```
Векторная операция (df["a"] + df["b"])    █ 1×        ⚡⚡⚡
.str / .dt аксессоры                       ██ 2-3×     ⚡⚡
np.where / np.select                       █ 1×        ⚡⚡⚡
Series.map(dict)                           ██ 2×       ⚡⚡
Series.apply(func)                         ████████ 8× ⚡
DataFrame.apply(axis=1)                    ████████████████ 50-100×  🐌
df.iterrows()                              ██████████████████████████ 200×+  🐌🐌
```

**Правило выбора (в порядке убывания приоритета):**

1. Векторная операция / NumPy
2. `np.where`, `np.select`, `pd.cut`, `pd.qcut`
3. `.map(dict)` — если задача про замену значений
4. `.str` / `.dt` аксессоры — если строки/даты
5. `Series.apply` — если ничего из выше не подходит
6. `DataFrame.apply(axis=1)` — последний ресурс
7. `iterrows()` — **никогда** в production


In [ ]:
# Бенчмарк: 4 способа сделать одно и то же
import time

np.random.seed(0)
n = 100_000
df = pd.DataFrame({"a": np.random.randint(1, 100, n), "b": np.random.randint(1, 100, n)})

# 1. Векторная
t0 = time.perf_counter()
df["c1"] = df["a"] + df["b"]
t_vec = time.perf_counter() - t0

# 2. Series.apply
t0 = time.perf_counter()
df["c2"] = df.apply(lambda r: r["a"] + r["b"], axis=1)
t_apply = time.perf_counter() - t0

# 3. iterrows
t0 = time.perf_counter()
res = []
for _, r in df.iterrows():
    res.append(r["a"] + r["b"])
df["c3"] = res
t_iter = time.perf_counter() - t0

print(f"Векторная:  {t_vec*1000:>8.1f} мс  (1×)")
print(f"apply:      {t_apply*1000:>8.1f} мс  ({t_apply/t_vec:.0f}×)")
print(f"iterrows:   {t_iter*1000:>8.1f} мс  ({t_iter/t_vec:.0f}×)")

---
## 18. Работа со строками (.str)

pandas предоставляет **строковый аксессор `.str`**: все методы Python-строк, работающие **поэлементно** на всей колонке. Это быстро и не требует `apply`.

Аксессор работает для колонок типа `object` и `string`. При `NaN` большинство методов возвращает `NaN`.


In [ ]:
df_str = pd.DataFrame({
    "full_name": ["  Иванов Иван  ", "петров пётр", "СИДОРОВА Елена"],
    "email":     ["ivan@corp.ru", "PETROV@mail.com", "elena_corp.ru"],
    "phone":     ["+7 999 111-22-33", "+7 (812) 333 44 55", "8-495-000-11-22"],
    "city":      ["Москва, Россия", "Санкт-Петербург", "Казань, Татарстан"],
})

In [ ]:
# Регистр
print(df_str["full_name"].str.lower())
print(df_str["full_name"].str.upper())
print(df_str["full_name"].str.title())

0      иванов иван  
1        петров пётр
2     сидорова елена
Name: full_name, dtype: object
0      ИВАНОВ ИВАН  
1        ПЕТРОВ ПЁТР
2     СИДОРОВА ЕЛЕНА
Name: full_name, dtype: object
0      Иванов Иван  
1        Петров Пётр
2     Сидорова Елена
Name: full_name, dtype: object


In [ ]:
# Убрать пробелы по краям — обязательный шаг очистки
df_str["full_name_clean"] = df_str["full_name"].str.strip()
print("До:    ", df_str["full_name"].tolist())
print("После: ", df_str["full_name_clean"].tolist())

До:     ['  Иванов Иван  ', 'петров пётр', 'СИДОРОВА Елена']
После:  ['Иванов Иван', 'петров пётр', 'СИДОРОВА Елена']


In [ ]:
# Проверки
print(df_str["email"].str.contains("@", na=False))
print(df_str["email"].str.startswith("ivan"))
print(df_str["email"].str.endswith(".ru"))

0     True
1     True
2    False
Name: email, dtype: bool
0     True
1    False
2    False
Name: email, dtype: bool
0     True
1    False
2     True
Name: email, dtype: bool


In [ ]:
# Замена — оставить только цифры в телефоне
df_str["phone_digits"] = df_str["phone"].str.replace(r"[^0-9]", "", regex=True)
print(df_str["phone_digits"])

0    79991112233
1    78123334455
2    84950001122
Name: phone_digits, dtype: object


In [ ]:
# Разделение строки
name_split = df_str["full_name_clean"].str.strip().str.split(r"\s+", n=1, expand=True)
df_str[["last_name","first_name"]] = name_split
df_str[["full_name_clean","last_name","first_name"]]

,full_name_clean,last_name,first_name
0,Иванов Иван,Иванов,Иван
1,петров пётр,петров,пётр
2,СИДОРОВА Елена,СИДОРОВА,Елена


In [ ]:
# Извлечение по regex
df_str["domain"]    = df_str["email"].str.extract(r"@(.+)$")
df_str["city_only"] = df_str["city"].str.extract(r"^([^,]+)")
df_str[["email","domain","city","city_only"]]

,email,domain,city,city_only
0,ivan@corp.ru,corp.ru,"Москва, Россия",Москва
1,PETROV@mail.com,mail.com,Санкт-Петербург,Санкт-Петербург
2,elena_corp.ru,NaN,"Казань, Татарстан",Казань


### 18.5 Дополнительные методы .str

#### .str.cat() — конкатенация колонок

Соединение нескольких строковых колонок в одну. С параметром `sep` — с разделителем. С параметром `na_rep` — что подставлять вместо NaN.


In [ ]:
df_n = pd.DataFrame({
    "first": ["Иван", "Пётр", None],
    "last":  ["Иванов", "Петров", "Сидоров"],
    "title": ["г-н", "г-н", "г-жа"],
})

# Через .str.cat — лучше, чем простое сложение, так как умеет обрабатывать NaN
df_n["full"] = df_n["title"].str.cat([df_n["first"], df_n["last"]], sep=" ", na_rep="?")
df_n

#### .str.zfill() / .str.pad() — выравнивание длины

Полезно для нормализации ID, артикулов, телефонов до фиксированной длины.


In [ ]:
ids = pd.Series([1, 23, 456, 7890])
print(ids.astype(str).str.zfill(6))   # ведущие нули до длины 6

# pad — то же, но с произвольным символом и стороной
names = pd.Series(["A", "BC", "DEF"])
print(names.str.pad(width=5, side="right", fillchar="."))

#### .str.slice() — срез строки

То же, что обычный срез `s.str[start:stop]`, но удобнее для цепочек.


In [ ]:
inn = pd.Series(["7710140679", "5040123456", "7825712345"])

# Первые 2 цифры — код региона
inn.str.slice(0, 2)
# Или эквивалент:
# inn.str[:2]

#### .str.findall() — извлечь все совпадения regex

Возвращает список найденных совпадений в каждой ячейке.


In [ ]:
text = pd.Series([
    "Контакты: ivan@mail.ru и petr@corp.ru",
    "Только один: anna@gmail.com",
    "Без почты вовсе",
])

# Все email-адреса в каждой строке
text.str.findall(r"[\w.]+@[\w.]+")

---
## 19. Группировка groupby

`groupby` — одна из ключевых возможностей pandas. Позволяет **разбить DataFrame на группы** по значениям колонок, **применить функцию** к каждой группе и **собрать результат**.

### Парадигма split → apply → combine

```
split:    [Иванов: [строки...], Петров: [строки...], Сидорова: [строки...]]
   ↓
apply:    sum(revenue) для каждой группы
   ↓
combine:  Иванов   -> 1 200 000
          Петров   ->   980 000
          Сидорова -> 1 450 000
```

### Стандартные агрегирующие функции

| Функция | Описание |
|---------|---------|
| `"sum"` | сумма |
| `"mean"` | среднее |
| `"median"` | медиана |
| `"count"` | кол-во непустых значений |
| `"size"` | кол-во строк включая NaN |
| `"min"` / `"max"` | минимум / максимум |
| `"std"` | стандартное отклонение |
| `"first"` / `"last"` | первое / последнее значение |
| `"nunique"` | кол-во уникальных значений |


In [ ]:
df.groupby("manager")["revenue"].sum()

KeyError: 'manager'

In [ ]:
df.groupby("manager")["revenue"].agg(["sum","mean","count","max"])

,sum,mean,count,max
manager,,,,
Иванов,3573357,7.146714e+05,5,1405416
Петров,7413049,1.235508e+06,6,1925665
Сидорова,9566408,1.062934e+06,9,1880488


### Named Aggregation — рекомендуемый стиль

Синтаксис `agg(имя=(источник, функция))` задаёт итоговые названия прямо при агрегации. Это чище, чем переименовывать потом.


In [201]:
result = df.groupby("manager").agg(
    revenue_sum  = ("revenue",     "sum"),
    revenue_mean = ("revenue",     "mean"),
    deals_count  = ("client_id",   "count"),
    margin_mean  = ("margin_rate", "mean"),
    orders_total = ("orders",      "sum"),
    top_deal     = ("revenue",     "max"),
).reset_index()

result["revenue_mean"] = result["revenue_mean"].round(0)
result["margin_mean"]  = result["margin_mean"].round(4)
result.sort_values("revenue_sum", ascending=False)

KeyError: 'manager'

In [ ]:
df.groupby(["manager","region"])["revenue"].agg(
    revenue_sum="sum", deals="count"
).reset_index().sort_values("revenue_sum", ascending=False)

,manager,region,revenue_sum,deals
2,Петров,Казань,6012478,5
6,Сидорова,СПб,4524079,4
4,Сидорова,Казань,4161636,3
0,Иванов,Москва,2606848,3
3,Петров,Москва,1400571,1
1,Иванов,СПб,966509,2
5,Сидорова,Москва,880693,2


### transform — результат той же длины

Обычный `groupby.agg()` сжимает DataFrame (одна строка на группу). `transform()` **возвращает Series той же длины**, что исходный DataFrame. Каждой строке присваивается значение её группы.

Применение: доля от группы, отклонение от среднего, нормализация внутри группы.


In [ ]:
df["manager_total"]  = df.groupby("manager")["revenue"].transform("sum")
df["deal_share_pct"] = (df["revenue"] / df["manager_total"] * 100).round(1)
df["vs_avg"]         = df["revenue"] - df.groupby("manager")["revenue"].transform("mean")

df[["manager","revenue","manager_total","deal_share_pct","vs_avg"]].head(8)

,manager,revenue,manager_total,deal_share_pct,vs_avg
0,Сидорова,431236,9566408,4.5,-631698.222222
1,Иванов,628178,3573357,17.6,-86493.400000
2,Сидорова,665894,9566408,7.0,-397040.222222
3,Сидорова,589492,9566408,6.2,-473442.222222
4,Иванов,1405416,3573357,39.3,690744.600000
5,Иванов,573254,3573357,16.0,-141417.400000
6,Сидорова,449457,9566408,4.7,-613477.222222
7,Петров,765987,7413049,10.3,-469521.166667


### filter — отбор групп по условию

`filter()` оставляет строки тех **групп**, которые удовлетворяют условию.

In [ ]:
df_top = df.groupby("manager").filter(
    lambda x: x["revenue"].sum() > 5_000_000
)
print("Менеджеры с выручкой > 5 млн:")
print(df_top["manager"].value_counts())

### 19.7 as_index=False — сохранение колонки группировки

По умолчанию `groupby` превращает колонку группировки в индекс. С `as_index=False` результат остаётся плоской таблицей.

```python
df.groupby("region").sum()                  # region -> индекс
df.groupby("region", as_index=False).sum()  # region остаётся колонкой
```

Это **эквивалент `.reset_index()`** после агрегации, но короче.


In [ ]:
np.random.seed(0)
df_g = pd.DataFrame({
    "region":  ["Москва","СПб","Москва","Казань","СПб"],
    "revenue": [100, 200, 150, 80, 220],
})

# С индексом
print("По умолчанию (region -> индекс):")
print(df_g.groupby("region")["revenue"].sum())
print()

# Без индекса
print("as_index=False:")
print(df_g.groupby("region", as_index=False)["revenue"].sum())

### 19.8 pd.Grouper — гибкая группировка по времени и значениям

`pd.Grouper` позволяет группировать по **временным интервалам** прямо в `groupby`, без `resample`. Полезно, когда нужно группировать одновременно по дате И по другой колонке.


In [ ]:
np.random.seed(0)
df_t = pd.DataFrame({
    "date":    pd.date_range("2026-01-01", periods=20, freq="D"),
    "region":  np.random.choice(["Москва","СПб"], 20),
    "revenue": np.random.randint(100, 500, 20),
})

# Группировка по неделям И по региону одновременно
result = df_t.groupby([
    pd.Grouper(key="date", freq="W"),
    "region"
])["revenue"].sum().reset_index()
result

### 19.9 get_group() — извлечение конкретной группы

`get_group(name)` возвращает подтаблицу одной группы как обычный DataFrame. Удобно для отладки и точечного анализа.


In [ ]:
groups = df_t.groupby("region")

# Все группы — их имена
print("Имена групп:", list(groups.groups.keys()))

# Одна группа целиком
moscow = groups.get_group("Москва")
print(f"\nКонкретно по Москве ({len(moscow)} строк):")
moscow.head()

### 19.10 nth() — N-я строка в каждой группе

`nth(0)` — первая строка группы, `nth(-1)` — последняя. Часто нужно при работе с временными рядами: «первая транзакция клиента», «последний платёж».


In [ ]:
np.random.seed(0)
df_p = pd.DataFrame({
    "client": ["A","A","A","B","B","C"],
    "date":   pd.to_datetime(["2026-01-05","2026-01-15","2026-02-10",
                              "2026-01-10","2026-03-01","2026-02-15"]),
    "amount": [100, 200, 150, 300, 250, 180],
}).sort_values(["client","date"])

# Первая транзакция каждого клиента
print("Первая транзакция:")
print(df_p.groupby("client").nth(0))

# Последняя
print("\nПоследняя транзакция:")
print(df_p.groupby("client").nth(-1))

---
## 20. Сводные таблицы

### 20.1 pivot_table — аналог сводной Excel

`pivot_table` — двумерная агрегация: одна переменная раскладывается по строкам, другая по столбцам.

Параметры:
- `values` — что агрегируем
- `index` — по строкам
- `columns` — по колонкам
- `aggfunc` — функция агрегации
- `fill_value` — заполнение отсутствующих комбинаций
- `margins=True` — строка/столбец с итогами


In [ ]:
pivot = pd.pivot_table(
    df,
    values      = "revenue",
    index       = "manager",
    columns     = "region",
    aggfunc     = "sum",
    fill_value  = 0,
    margins     = True,
    margins_name= "Итого"
)
pivot

KeyError: 'manager'

In [ ]:
# Несколько значений и функций
pivot2 = pd.pivot_table(
    df,
    values  = ["revenue","margin"],
    index   = "manager",
    columns = "region",
    aggfunc = {"revenue": "sum", "margin": "mean"},
    fill_value = 0,
)
pivot2

### 20.2 crosstab — таблица частот и сопряжённости

`pd.crosstab()` считает количество вхождений каждой комбинации двух категориальных переменных. С `normalize` — доли вместо счётчиков.


In [ ]:
pd.crosstab(df["region"], df["client_size"])

In [ ]:
# Доли по строкам (% каждого размера клиента в регионе)
pd.crosstab(
    df["region"], df["client_size"],
    normalize="index"   # "index"=по строкам, "columns"=по столбцам, "all"=от итого
).round(2)

### 20.3 pivot() vs pivot_table() — в чём разница?

| Аспект | `pivot()` | `pivot_table()` |
|--------|-----------|----------------|
| Дубли в индексе | ❌ падает с ошибкой | ✅ агрегирует |
| Агрегация | нет | да (`aggfunc`) |
| `fill_value` | нет | да |
| `margins` | нет | да |
| Использование | reshape без агрегации | сводная с агрегацией |

**Правило:** если в данных есть дубли по комбинации (index, columns) — нужен `pivot_table`. Если все комбинации уникальны — `pivot` чуть быстрее.


In [ ]:
# С pivot — нужны уникальные комбинации
df_unique = pd.DataFrame({
    "manager": ["А","А","Б","Б"],
    "month":   ["jan","feb","jan","feb"],
    "revenue": [100, 120, 200, 180],
})

print("pivot() работает:")
print(df_unique.pivot(index="manager", columns="month", values="revenue"))

In [ ]:
# Если есть дубли — pivot() упадёт, нужно pivot_table()
df_dups = pd.DataFrame({
    "manager": ["А","А","А","Б"],
    "month":   ["jan","jan","feb","jan"],   # А-jan встречается дважды!
    "revenue": [100, 50, 120, 200],
})

# pivot() выбросит ошибку:
# df_dups.pivot(...)  # ValueError: Index contains duplicate entries

# pivot_table() агрегирует
print(pd.pivot_table(df_dups, index="manager", columns="month",
                     values="revenue", aggfunc="sum"))

### 20.4 Многоуровневые сводные таблицы

Если в `index` или `columns` передать **список** колонок — получится многоуровневый pivot.


In [202]:
np.random.seed(0)
df_pv = pd.DataFrame({
    "region":   np.random.choice(["Москва","СПб"], 20),
    "channel":  np.random.choice(["online","offline"], 20),
    "month":    np.random.choice(["jan","feb","mar"], 20),
    "revenue":  np.random.randint(100, 500, 20),
})

# Двухуровневые строки
pd.pivot_table(
    df_pv,
    values="revenue",
    index=["region","channel"],   # два уровня в строках
    columns="month",
    aggfunc="sum",
    fill_value=0
)

month           feb  jan  mar
region channel               
Москва offline    0  831  376
       online   373  788  489
СПб    offline  822  626  357
       online   484  476  826

---
## 21. Объединение таблиц

### 21.1 concat — склейка по осям

`concat` **укладывает** DataFrame'ы друг за другом — по строкам (axis=0) или по колонкам (axis=1). Не ищет совпадений по ключам. Используется когда структура одинакова: объединить данные за несколько месяцев.


In [ ]:
df_jan = df.iloc[:7].copy()
df_feb = df.iloc[7:14].copy()
df_mar = df.iloc[14:].copy()

df_all = pd.concat([df_jan, df_feb, df_mar], ignore_index=True)
print(f"Склеено строк: {len(df_all)}")

# По колонкам
df_wide = pd.concat([df[["manager","region"]], df[["revenue","cost"]]], axis=1)
df_wide.head(3)

### 21.2 merge — SQL-стиль JOIN

`merge` объединяет таблицы по **общему ключу**. Это прямой аналог SQL JOIN.

### Типы объединения

| how | SQL | Результат |
|-----|-----|----------|
| `"left"` | LEFT JOIN | все из левой + совпадения из правой |
| `"right"` | RIGHT JOIN | совпадения из левой + все из правой |
| `"inner"` | INNER JOIN | только совпадающие ключи |
| `"outer"` | FULL OUTER JOIN | все из обеих таблиц |

**В аналитике чаще всего `left`** — нужно сохранить все транзакции и добавить данные из справочника.


In [ ]:
df_orders = pd.DataFrame({
    "order_id":  [1, 2, 3, 4, 5, 6],
    "client_id": [101, 102, 101, 103, 104, 102],  # 104 — нет в справочнике
    "amount":    [50_000, 120_000, 30_000, 200_000, 75_000, 90_000],
})

df_clients = pd.DataFrame({
    "client_id": [101, 102, 103],
    "name":      ["ООО Альфа", "ООО Бета", "ООО Гамма"],
    "segment":   ["крупный", "средний", "крупный"],
})

# LEFT JOIN: все заказы, клиент 104 -> NaN в name и segment
df_merged = df_orders.merge(df_clients, on="client_id", how="left")
df_merged

,order_id,client_id,amount,name,segment
0,1,101,50000,ООО Альфа,крупный
1,2,102,120000,ООО Бета,средний
2,3,101,30000,ООО Альфа,крупный
3,4,103,200000,ООО Гамма,крупный
4,5,104,75000,NaN,NaN
5,6,102,90000,ООО Бета,средний


In [ ]:
# Разные названия ключей
df_orders.merge(
    df_clients.rename(columns={"client_id": "id"}),
    left_on="client_id", right_on="id", how="left"
)

,order_id,client_id,amount,id,name,segment
0,1,101,50000,101.0,ООО Альфа,крупный
1,2,102,120000,102.0,ООО Бета,средний
2,3,101,30000,101.0,ООО Альфа,крупный
3,4,103,200000,103.0,ООО Гамма,крупный
4,5,104,75000,NaN,NaN,NaN
5,6,102,90000,102.0,ООО Бета,средний


### 21.3 Проверка качества merge — обязательный шаг

После merge проверяйте: не появились ли лишние строки (дубли ключей) и не потерялись ли строки (нет совпадений).


In [ ]:
print("ДО merge:", df_orders.shape)

df_check = df_orders.merge(
    df_clients, on="client_id", how="left",
    indicator=True        # добавляет колонку _merge
)

print("ПОСЛЕ merge:", df_check.shape)
print()
print(df_check["_merge"].value_counts())
print()
df_check[df_check["_merge"] == "left_only"]   # строки без пары

ДО merge: (6, 3)
ПОСЛЕ merge: (6, 6)

_merge
both          5
left_only     1
right_only    0
Name: count, dtype: int64



,order_id,client_id,amount,name,segment,_merge
4,5,104,75000,NaN,NaN,left_only


In [ ]:
# validate проверяет тип связи — выбросит ошибку при нарушении
try:
    df_orders.merge(df_clients, on="client_id", how="left", validate="many_to_one")
    print("✅ Тип связи many_to_one подтверждён")
except Exception as e:
    print(f"❌ Ошибка: {e}")

✅ Тип связи many_to_one подтверждён


### 21.4 suffixes — что делать, если колонки совпадают

Когда обе таблицы содержат колонки с одинаковыми именами (кроме ключа JOIN), pandas добавляет к ним суффиксы. По умолчанию это `"_x"` и `"_y"`, что выглядит неинформативно.


In [ ]:
# Обе таблицы имеют колонку "revenue"
df_q1 = pd.DataFrame({"client_id": [1, 2, 3], "revenue": [100, 200, 150]})
df_q2 = pd.DataFrame({"client_id": [1, 2, 3], "revenue": [120, 180, 200]})

# По умолчанию — _x, _y
print(df_q1.merge(df_q2, on="client_id"))
print()
# Свои суффиксы
print(df_q1.merge(df_q2, on="client_id", suffixes=("_q1", "_q2")))

### 21.5 merge_asof() — для нечётких соответствий по дате

Уникальная фича pandas. `merge_asof` объединяет таблицы по ключу, но **не требует точного совпадения** — берёт ближайшее значение. Это незаменимо для финансовых данных: «прикрепить курс валюты на дату транзакции».


In [ ]:
# Транзакции
trans = pd.DataFrame({
    "ts":     pd.to_datetime(["2026-01-15 10:00","2026-01-15 14:00",
                               "2026-01-16 09:00","2026-01-17 12:00"]),
    "amount": [1000, 2000, 1500, 800]
})

# Курсы валют (обновляются раз в день)
rates = pd.DataFrame({
    "ts":   pd.to_datetime(["2026-01-15 00:00","2026-01-16 00:00",
                            "2026-01-17 00:00"]),
    "rate": [90.5, 91.2, 89.8]
})

# Для каждой транзакции — курс на момент или ранее
result = pd.merge_asof(
    trans.sort_values("ts"),
    rates.sort_values("ts"),
    on="ts",
    direction="backward"   # брать последнее значение до этого момента
)
result["amount_usd"] = (result["amount"] / result["rate"]).round(2)
result

### 21.6 combine_first() — заполнение пропусков из другой таблицы

`combine_first(other)` берёт значения из `self`, а где они `NaN` — подставляет из `other`. Это «мягкое слияние» для перекрытия неполных данных.


In [ ]:
# Основная таблица с пропусками
df_main = pd.DataFrame({
    "client_id": [1, 2, 3, 4],
    "email":     ["a@x.ru", None, "c@x.ru", None],
}).set_index("client_id")

# Запасной источник
df_backup = pd.DataFrame({
    "client_id": [1, 2, 3, 4],
    "email":     ["a_old@x.ru", "b@x.ru", "c_old@x.ru", "d@x.ru"],
}).set_index("client_id")

# Берём из основной, где есть; иначе — из запасной
df_main.combine_first(df_backup)

### 21.7 join() — упрощённый merge через индекс

`join()` — обёртка над `merge` для объединения по индексу. Краткий синтаксис, когда правая таблица — это «справочник» с информативным индексом.


In [ ]:
orders = pd.DataFrame({
    "order_id":  [1, 2, 3, 4],
    "client_id": [101, 102, 101, 103],
    "amount":    [100, 200, 50, 300],
}).set_index("client_id")

clients = pd.DataFrame({
    "name":    ["Альфа", "Бета", "Гамма"],
    "segment": ["VIP", "Стандарт", "VIP"],
}, index=[101, 102, 103])
clients.index.name = "client_id"

# LEFT JOIN по индексу — лаконично
orders.join(clients)

---
## 22. Изменение формы таблицы

### Wide vs Long форматы

**Wide (широкий)** — каждый период/категория = отдельная колонка:
```
manager | jan     | feb     | mar
Иванов  | 100 000 | 120 000 | 130 000
```

**Long (длинный)** — каждое наблюдение = отдельная строка:
```
manager | month | revenue
Иванов  | jan   | 100 000
Иванов  | feb   | 120 000
```

**Long-формат** — универсальный для аналитики: подходит для `groupby`, `pivot_table`, построения графиков. **Wide-формат** удобен для чтения человеком и Excel.

### 22.1 melt — wide → long


In [ ]:
df_wide = pd.DataFrame({
    "manager": ["Иванов","Петров","Сидорова"],
    "jan": [100_000, 200_000, 150_000],
    "feb": [120_000, 180_000, 160_000],
    "mar": [130_000, 220_000, 140_000],
})

df_long = df_wide.melt(
    id_vars    = ["manager"],
    value_vars = ["jan","feb","mar"],
    var_name   = "month",
    value_name = "revenue"
)
df_long

### 22.2 pivot — long → wide

In [ ]:
df_back = df_long.pivot(
    index="manager", columns="month", values="revenue"
).reset_index()
df_back.columns.name = None
df_back

### 22.3 stack и unstack

`stack()` — переносит колонки **в индекс**. `unstack()` — переносит уровень индекса **в колонки**.

In [ ]:
stacked = df_back.set_index("manager").stack()
print(stacked.head(6))
stacked.unstack().head()

### 22.4 Когда какой инструмент reshape выбирать

| Задача | Инструмент |
|--------|------------|
| Уникальные комбинации, без агрегации | `df.pivot(...)` |
| С дублями или агрегацией | `pd.pivot_table(...)` |
| Wide → Long, один уровень | `df.melt(...)` |
| Wide → Long, сложные паттерны имён | `pd.wide_to_long(...)` |
| MultiIndex колонки → строки | `df.stack()` |
| MultiIndex строки → колонки | `df.unstack()` |
| Список в ячейке → строки | `df.explode(...)` |

### pd.wide_to_long — структурированный wide-to-long

Если у вас данные с колонками типа `revenue_2024`, `revenue_2025`, `cost_2024`, `cost_2025` — обычный `melt` не вытащит структуру. Для таких случаев есть специальная функция `wide_to_long`.


In [ ]:
df_w = pd.DataFrame({
    "manager":     ["А", "Б", "В"],
    "revenue_2024":[100, 200, 150],
    "revenue_2025":[120, 220, 180],
    "cost_2024":   [50,  100, 80],
    "cost_2025":   [60,  110, 95],
})

# Преобразуем: stubnames - префиксы, j - имя новой колонки с числом
result = pd.wide_to_long(
    df_w,
    stubnames=["revenue", "cost"],   # префиксы для группировки
    i="manager",                       # уникальный ключ
    j="year",                          # имя для года
    sep="_"                            # разделитель в названиях
).reset_index()
result

---
## 23. Даты и время

Работа с датами — одна из самых частых задач в аналитике. pandas предоставляет мощный инструментарий: специальные типы данных, аксессор `.dt`, парсер `pd.to_datetime`, временные зоны, периоды и многое другое.

### Карта типов для работы со временем

```
Понятие                  Тип pandas           Назначение
─────────────────────────────────────────────────────────────────────
Точка во времени         Timestamp            конкретный момент: дата сделки
Массив точек             DatetimeIndex        ось времени, индекс DataFrame
Интервал времени         Timedelta            разность дат: длительность
Массив интервалов        TimedeltaIndex       массив длительностей
Календарный период       Period               отчётный период: "Q1 2026"
Массив периодов          PeriodIndex          серия периодов
Смещение (бизнес-лог.)   DateOffset           умный сдвиг: +1 месяц, +5 рабочих дней
```

### Когда что использовать

| Тип | Применение |
|-----|-----------|
| `Timestamp` / `datetime64` | Конкретный момент: дата транзакции, дата регистрации |
| `Timedelta` | Разница между двумя моментами: длительность сделки, возраст |
| `Period` | Временно́й период без точного момента: «январь 2026», «Q1 2026» |
| `DateOffset` | Сдвиг по бизнес-логике: следующий рабочий день, конец месяца |

### Почему важно хранить даты как datetime64, а не строки

| Тип | Что можно | Что нельзя |
|-----|-----------|-----------|
| `object` (строка) | сравнивать как строки | арифметика, `.dt`, `resample` |
| `datetime64` | всё: год/месяц, разность, resample, фильтрация | — |

**Первый шаг при загрузке данных с датами:** `pd.to_datetime(df["date"], errors="coerce")`


In [ ]:
import numpy as np
import pandas as pd

# Посмотрим на все временны́е типы
ts  = pd.Timestamp("2026-03-15 14:30:00")
td  = pd.Timedelta("3 days 4 hours")
per = pd.Period("2026-Q1", freq="Q")
off = pd.DateOffset(months=1)

print(f"Timestamp:   {ts}   | тип: {type(ts).__name__}")
print(f"Timedelta:   {td}  | тип: {type(td).__name__}")
print(f"Period:      {per}          | тип: {type(per).__name__}")
print(f"DateOffset:  {off}       | тип: {type(off).__name__}")

### 23.1 Timestamp — точка во времени

`pd.Timestamp` — базовый объект для дат в pandas. Это аналог Python `datetime.datetime`, но с расширенным функционалом и точностью до наносекунд.

Когда вы преобразуете колонку через `pd.to_datetime()`, каждая ячейка становится объектом `Timestamp`. При работе с отдельными значениями (не колонками) вы тоже работаете с `Timestamp`.

Принимает множество форматов: строку ISO (`"2026-03-15"`), произвольную строку (`"15.03.2026"`), компоненты по отдельности, Python `datetime`, NumPy `datetime64`.


In [ ]:
# Создание разными способами
ts1 = pd.Timestamp("2026-03-15")
ts2 = pd.Timestamp("2026-03-15 14:30:00")
ts3 = pd.Timestamp("15.03.2026")
ts4 = pd.Timestamp(year=2026, month=3, day=15, hour=9)

print(ts1)
print(ts2)
print(ts3)
print(ts4)

In [ ]:
# Текущий момент
print(f"Сейчас:  {pd.Timestamp.now()}")
print(f"Сегодня: {pd.Timestamp.today().date()}")

In [ ]:
# Атрибуты Timestamp
ts = pd.Timestamp("2026-03-15 14:30:45.123456")

print(f"year:        {ts.year}")
print(f"month:       {ts.month}")
print(f"day:         {ts.day}")
print(f"hour:        {ts.hour}")
print(f"minute:      {ts.minute}")
print(f"second:      {ts.second}")
print(f"microsecond: {ts.microsecond}")
print(f"weekday():   {ts.weekday()}  (0=Пн, 6=Вс)")
print(f"day_name():  {ts.day_name()}")
print(f"month_name():{ts.month_name()}")
print(f"quarter:     {ts.quarter}")
print(f"day_of_year: {ts.day_of_year}")
print(f"week:        {ts.isocalendar().week}")

year:        2026
month:       3
day:         15
hour:        14
minute:      30
second:      45
microsecond: 123456
weekday():   6  (0=Пн, 6=Вс)
day_name():  Sunday
month_name():March
quarter:     1
day_of_year: 74
week:        11


In [ ]:
# Арифметика с Timestamp
ts = pd.Timestamp("2026-03-15")

print(ts + pd.Timedelta(days=10))         # +10 дней
print(ts - pd.Timedelta(weeks=2))         # -2 недели
print(ts + pd.DateOffset(months=1))       # +1 месяц (умный сдвиг)
print(ts + pd.DateOffset(years=1))        # +1 год

# Разница двух Timestamp -> Timedelta
delta = pd.Timestamp("2026-06-30") - ts
print(f"\nРазница: {delta}, дней: {delta.days}")

In [ ]:
# Преобразование в другие типы
ts = pd.Timestamp("2026-03-15 14:30:00")

print(ts.to_pydatetime())   # Python datetime
print(ts.to_numpy())        # NumPy datetime64
print(ts.timestamp())       # Unix timestamp (секунды с 1970-01-01)
print(ts.date())            # Python date (без времени)

### 23.2 Создание последовательностей дат

`pd.date_range()` — основной инструмент для создания временных осей, тестовых данных и заполнения пропусков в рядах. Возвращает `DatetimeIndex`.

Параметры:
- `start` — начало
- `end` — конец (включительно)
- `periods` — количество точек (вместо `end`)
- `freq` — шаг (алиас)

### Алиасы частот (freq)

| Алиас | Период |
|-------|--------|
| `"D"` | каждый день |
| `"B"` | рабочие дни (Пн–Пт) |
| `"W"` | каждое воскресенье |
| `"ME"` | конец месяца |
| `"MS"` | начало месяца |
| `"QE"` | конец квартала |
| `"QS"` | начало квартала |
| `"YE"` | 31 декабря |
| `"YS"` | 1 января |
| `"h"` | каждый час |
| `"min"` | каждую минуту |
| `"s"` | каждую секунду |
| `"2D"` | число перед алиасом = множитель (каждые 2 дня) |


In [ ]:
# По start + end
idx = pd.date_range(start="2026-01-01", end="2026-01-10", freq="D")
print(idx)

In [ ]:
# По start + periods
idx = pd.date_range(start="2026-01-01", periods=12, freq="ME")
print(idx)

In [ ]:
# Разные частоты
print("Рабочие дни:")
print(pd.date_range("2026-01-01", periods=5, freq="B"))

print("\nНачало месяца:")
print(pd.date_range("2026-01-01", periods=6, freq="MS"))

print("\nКонец квартала:")
print(pd.date_range("2026-01-01", periods=4, freq="QE"))

print("\nКаждые 2 дня:")
print(pd.date_range("2026-01-01", periods=5, freq="2D"))

In [ ]:
# Генерация тестовых данных с date_range
np.random.seed(42)
df = pd.DataFrame({
    "date":    pd.date_range("2026-01-01", periods=30, freq="D"),
    "revenue": np.random.randint(100_000, 500_000, 30),
    "manager": np.random.choice(["Иванов","Петров","Сидорова"], 30),
})
df.head()

**Связанные функции:**
- `pd.period_range()` — последовательность периодов (Period)
- `pd.timedelta_range()` — последовательность интервалов (Timedelta)


In [ ]:
# Месячные периоды
months = pd.period_range("2026-01", periods=6, freq="M")
print(months)

# Квартальные периоды
quarters = pd.period_range("2025Q1", periods=6, freq="Q")
print(quarters)

# Интервалы
intervals = pd.timedelta_range(start="0 days", periods=5, freq="6h")
print(intervals)

### 23.3 Парсинг дат: pd.to_datetime

`pd.to_datetime()` — главный инструмент для преобразования строк, чисел и других типов в `datetime64`. Это **первое, что делают** с датами при загрузке данных.

### Параметры

| Параметр | Назначение |
|----------|-----------|
| `arg` | что преобразуем: строка, Series, список, DataFrame |
| `errors` | `"raise"` (ошибка), `"coerce"` (→ NaT), `"ignore"` |
| `format` | явный формат — **ускоряет** парсинг в разы |
| `dayfirst` | `True` если дата в формате DD.MM.YYYY |
| `unit` | для числовых меток: `"s"`, `"ms"`, `"us"`, `"ns"` |
| `utc` | конвертировать в UTC |

### Правило errors="coerce"

В реальных данных всегда есть строки, которые не являются датой: пустые ячейки, «нет данных», опечатки. `errors="coerce"` безопасно конвертирует их в `NaT` (Not a Time — аналог `NaN` для дат) вместо того, чтобы падать с ошибкой.


In [ ]:
# Базовый парсинг
dates = pd.Series(["2026-01-15", "2026-02-28", "2026-03-10"])
print(pd.to_datetime(dates))
print(pd.to_datetime(dates).dtype)

In [ ]:
# Разные форматы строк
formats = pd.Series([
    "2026-01-15",           # ISO
    "15/01/2026",           # DD/MM/YYYY
    "January 15, 2026",     # текстовый формат
    "15.01.2026",           # DD.MM.YYYY
    "20260115",             # компактный YYYYMMDD
])
print(pd.to_datetime(formats, dayfirst=True))

In [ ]:
# errors="coerce" — невалидные значения становятся NaT
messy = pd.Series(["2026-01-15", "нет даты", "", "2026-03-10", "???"])
result = pd.to_datetime(messy, errors="coerce")
print(result)
print(f"\nNaT (пропусков): {result.isna().sum()}")

In [ ]:
# Явный формат — намного быстрее на больших данных!
ru_dates = pd.Series(["15.01.2026", "28.02.2026", "10.03.2026"])

# Медленно — автоопределение
slow = pd.to_datetime(ru_dates, dayfirst=True)

# Быстро — явный формат
fast = pd.to_datetime(ru_dates, format="%d.%m.%Y")

print(fast)
print("Результат идентичен:", slow.equals(fast))

### Коды формата strftime/strptime

| Код | Описание | Пример |
|-----|---------|--------|
| `%Y` | год 4 цифры | `2026` |
| `%y` | год 2 цифры | `26` |
| `%m` | месяц 01–12 | `03` |
| `%d` | день 01–31 | `15` |
| `%H` | часы 00–23 | `14` |
| `%M` | минуты 00–59 | `30` |
| `%S` | секунды 00–59 | `45` |
| `%f` | микросекунды | `123456` |
| `%p` | AM/PM | `PM` |
| `%A` | полное название дня | `Monday` |
| `%B` | полное название месяца | `March` |
| `%j` | день года 001–366 | `074` |
| `%W` | номер недели | `11` |


In [ ]:
# Форматы с временем
datetime_strings = pd.Series([
    "15.03.2026 14:30:00",
    "28.02.2026 09:00:00",
    "10.01.2026 23:59:59",
])
result = pd.to_datetime(datetime_strings, format="%d.%m.%Y %H:%M:%S")
print(result)

In [ ]:
# Парсинг из Unix timestamp (секунды с 1970-01-01)
unix_ts = pd.Series([1_700_000_000, 1_710_000_000, 1_720_000_000])
print(pd.to_datetime(unix_ts, unit="s"))
# unit="ms" для миллисекунд, "us" для микросекунд

In [ ]:
# Парсинг из нескольких колонок
df_parts = pd.DataFrame({
    "year":  [2026, 2026, 2025],
    "month": [1, 3, 12],
    "day":   [15, 10, 31],
})
df_parts["date"] = pd.to_datetime(df_parts[["year","month","day"]])
df_parts

In [ ]:
# Атрибуты Timestamp
ts = pd.Timestamp("2026-03-15 14:30:45.123456")

print(f"year:        {ts.year}")
print(f"month:       {ts.month}")
print(f"day:         {ts.day}")
print(f"hour:        {ts.hour}")
print(f"minute:      {ts.minute}")
print(f"second:      {ts.second}")
print(f"microsecond: {ts.microsecond}")
print(f"weekday():   {ts.weekday()}  (0=Пн, 6=Вс)")
print(f"day_name():  {ts.day_name()}")
print(f"month_name():{ts.month_name()}")
print(f"quarter:     {ts.quarter}")
print(f"day_of_year: {ts.day_of_year}")
print(f"week:        {ts.isocalendar().week}")

year:        2026
month:       3
day:         15
hour:        14
minute:      30
second:      45
microsecond: 123456
weekday():   6  (0=Пн, 6=Вс)
day_name():  Sunday
month_name():March
quarter:     1
day_of_year: 74
week:        11


In [ ]:
# Практика: очистка реальной колонки
df_raw = pd.DataFrame({
    "deal_id":   [1, 2, 3, 4, 5],
    "date_str":  ["2026-01-15", "нет", "2026-03-10", "", "2026-04-01"],
    "close_str": ["15.02.2026", "28.03.2026", "20.04.2026", "нет", "30.06.2026"],
    "revenue":   [100_000, 250_000, 175_000, 80_000, 300_000],
})

df_raw["date"]  = pd.to_datetime(df_raw["date_str"],  errors="coerce")
df_raw["close"] = pd.to_datetime(df_raw["close_str"], errors="coerce", dayfirst=True)

print("Пропуски после парсинга:")
print(df_raw[["date","close"]].isna().sum())
df_raw[["deal_id","date","close","revenue"]]

### 23.4 Аксессор .dt — полный справочник

Аксессор `.dt` открывает доступ ко всем свойствам и методам datetime для **целой колонки** сразу — без циклов, без `apply`. Это быстро и лаконично.

> **Важно:** `.dt` работает только с колонками типа `datetime64`. Если колонка — строка, сначала выполните `pd.to_datetime()`.

### Группы атрибутов и методов

```
.dt
├── Компоненты даты    year, month, day, quarter, day_of_year, day_of_week
├── Компоненты времени hour, minute, second, microsecond
├── Имена              day_name(), month_name()
├── Флаги              is_month_start, is_month_end, is_quarter_start...
├── ISO-календарь      isocalendar() → year, week, day
├── Округление         floor(), ceil(), round()
├── Форматирование     strftime()
├── Период             to_period()
└── Unix               timestamp()
```


In [ ]:
# Тестовый DataFrame с датами
np.random.seed(10)
df_dt = pd.DataFrame({
    "deal_id":  range(1, 13),
    "date":     pd.date_range("2026-01-05", periods=12, freq="ME"),
    "revenue":  np.random.randint(100_000, 1_500_000, 12),
})
df_dt.head()

**Компоненты даты**

In [ ]:
df_dt["year"]         = df_dt["date"].dt.year
df_dt["month"]        = df_dt["date"].dt.month
df_dt["day"]          = df_dt["date"].dt.day
df_dt["quarter"]      = df_dt["date"].dt.quarter
df_dt["day_of_year"]  = df_dt["date"].dt.day_of_year   # 1–366
df_dt["day_of_week"]  = df_dt["date"].dt.day_of_week   # 0=Пн, 6=Вс
df_dt["days_in_month"]= df_dt["date"].dt.days_in_month  # сколько дней в месяце

df_dt[["date","year","month","day","quarter","day_of_year","day_of_week","days_in_month"]]

NameError: name 'df_dt' is not defined

**Компоненты времени**

In [ ]:
df_dt["datetime"] = pd.date_range("2026-01-05 09:15:30", periods=12, freq="45min")

df_dt["hour"]        = df_dt["datetime"].dt.hour
df_dt["minute"]      = df_dt["datetime"].dt.minute
df_dt["second"]      = df_dt["datetime"].dt.second

df_dt[["datetime","hour","minute","second"]].head(6)

**Названия дней и месяцев — в т.ч. на русском**

In [ ]:
df_dt["weekday_name"] = df_dt["date"].dt.day_name()
df_dt["month_name"]   = df_dt["date"].dt.month_name()

# Локализация на русский — через словарь
weekday_ru = {
    "Monday":"Понедельник","Tuesday":"Вторник","Wednesday":"Среда",
    "Thursday":"Четверг","Friday":"Пятница","Saturday":"Суббота","Sunday":"Воскресенье"
}
month_ru = {
    "January":"Январь","February":"Февраль","March":"Март","April":"Апрель",
    "May":"Май","June":"Июнь","July":"Июль","August":"Август",
    "September":"Сентябрь","October":"Октябрь","November":"Ноябрь","December":"Декабрь"
}

df_dt["weekday_ru"] = df_dt["date"].dt.day_name().map(weekday_ru)
df_dt["month_ru"]   = df_dt["date"].dt.month_name().map(month_ru)

df_dt[["date","weekday_ru","month_ru"]].head(6)

**Флаги начала/конца периодов**

In [ ]:
df_dt["is_month_start"]   = df_dt["date"].dt.is_month_start
df_dt["is_month_end"]     = df_dt["date"].dt.is_month_end
df_dt["is_quarter_start"] = df_dt["date"].dt.is_quarter_start
df_dt["is_quarter_end"]   = df_dt["date"].dt.is_quarter_end
df_dt["is_year_start"]    = df_dt["date"].dt.is_year_start
df_dt["is_year_end"]      = df_dt["date"].dt.is_year_end
df_dt["is_leap_year"]     = df_dt["date"].dt.is_leap_year

df_dt[["date","is_month_start","is_month_end","is_quarter_end","is_year_end"]].head(6)

**ISO-календарь** (год, номер недели, день недели 1=Пн..7=Вс)

In [ ]:
iso = df_dt["date"].dt.isocalendar()
print(iso.head(6))

df_dt["iso_year"] = iso["year"]
df_dt["iso_week"] = iso["week"]
df_dt["iso_day"]  = iso["day"]
df_dt[["date","iso_year","iso_week","iso_day"]].head(6)

**Округление: floor, ceil, round**

In [ ]:
df_dt["ts"] = pd.date_range("2026-01-01 08:37:52", periods=12, freq="137min")

df_dt["floor_hour"]  = df_dt["ts"].dt.floor("h")   # вниз до часа
df_dt["ceil_hour"]   = df_dt["ts"].dt.ceil("h")    # вверх до часа
df_dt["round_hour"]  = df_dt["ts"].dt.round("h")   # к ближайшему часу
df_dt["round_15min"] = df_dt["ts"].dt.round("15min")

df_dt[["ts","floor_hour","ceil_hour","round_hour","round_15min"]].head(5)

**strftime — форматирование дат в строки**

In [ ]:
df_dt["date_str"]       = df_dt["date"].dt.strftime("%d.%m.%Y")
df_dt["month_year_str"] = df_dt["date"].dt.strftime("%B %Y")
df_dt["year_month_key"] = df_dt["date"].dt.strftime("%Y-%m")
df_dt["quarter_label"]  = "Q" + df_dt["date"].dt.quarter.astype(str) + " " +                           df_dt["date"].dt.year.astype(str)

df_dt[["date","date_str","month_year_str","year_month_key","quarter_label"]].head(6)

**to_period — преобразование в Period**

In [ ]:
df_dt["period_month"]   = df_dt["date"].dt.to_period("M")
df_dt["period_quarter"] = df_dt["date"].dt.to_period("Q")
df_dt["period_year"]    = df_dt["date"].dt.to_period("Y")

df_dt[["date","period_month","period_quarter","period_year"]].head(6)

In [ ]:
# Period -> Timestamp (начало или конец периода)
df_dt["month_start"] = df_dt["date"].dt.to_period("M").dt.to_timestamp()
df_dt["month_end"]   = df_dt["date"].dt.to_period("M").dt.to_timestamp("M")
df_dt[["date","month_start","month_end"]].head(6)

**date() и time() — только дата или только время**

In [ ]:
df_x = pd.DataFrame({
    "ts": pd.date_range("2026-01-15 09:30:00", periods=5, freq="75min")
})
df_x["only_date"] = df_x["ts"].dt.date
df_x["only_time"] = df_x["ts"].dt.time
df_x

### 23.5 Timedelta — работа с интервалами времени

`Timedelta` представляет **разность между двумя моментами** — длительность. Это pandas-аналог Python `datetime.timedelta`.

### Откуда берётся Timedelta

Чаще всего возникает автоматически при вычитании двух datetime-колонок:

```python
df["duration"] = df["close_date"] - df["open_date"]   # -> Timedelta
```

### Единицы измерения

| Параметр | Значение |
|----------|---------|
| `weeks` | недели |
| `days` | дни |
| `hours` | часы |
| `minutes` | минуты |
| `seconds` | секунды |
| `milliseconds` | миллисекунды |
| `microseconds` | микросекунды |
| `nanoseconds` | наносекунды |


In [ ]:
# Создание Timedelta разными способами
td1 = pd.Timedelta("3 days")
td2 = pd.Timedelta("1 day 4 hours 30 minutes")
td3 = pd.Timedelta(days=5, hours=3)
td4 = pd.Timedelta(weeks=2)

print(td1)
print(td2)
print(td3)
print(td4)

In [ ]:
# Атрибуты
td = pd.Timedelta("3 days 4 hours 30 minutes 15 seconds")

print(f"days:           {td.days}")
print(f"seconds:        {td.seconds}")          # секунды после дней
print(f"total_seconds:  {td.total_seconds()}")  # всё в секундах
print(f"components:")
print(td.components)

In [ ]:
# Разница между датами -> Timedelta Series
df_deals = pd.DataFrame({
    "deal_id":    [1, 2, 3, 4, 5],
    "open_date":  pd.to_datetime(["2026-01-05","2026-01-12","2026-02-01",
                                   "2026-02-15","2026-03-01"]),
    "close_date": pd.to_datetime(["2026-02-10","2026-01-20","2026-02-28",
                                   "2026-04-01","2026-03-25"]),
})

df_deals["duration"]       = df_deals["close_date"] - df_deals["open_date"]
df_deals["duration_days"]  = df_deals["duration"].dt.days
df_deals["duration_weeks"] = df_deals["duration"].dt.days // 7

df_deals

In [ ]:
# Извлечение компонентов из Timedelta колонки
td_series = df_deals["duration"]

print("Дни:")
print(td_series.dt.days.tolist())

print("\nTotal секунд:")
print(td_series.dt.total_seconds().astype(int).tolist())

print("\nКомпоненты:")
print(td_series.dt.components)

In [ ]:
# Арифметика
td = pd.Timedelta(days=10)
ts = pd.Timestamp("2026-03-01")

print(ts + td)               # Timestamp + Timedelta = Timestamp
print(ts - td)
print(td + pd.Timedelta(days=5))   # Timedelta + Timedelta
print(td / 2)                # / число
print(td * 3)                # * число

# Сравнение
print(td > pd.Timedelta(days=5))     # True

In [ ]:
# Классификация сроков
df_deals["status"] = pd.cut(
    df_deals["duration_days"],
    bins   = [0, 14, 30, 60, 999],
    labels = ["быстрая (≤2 нед)","нормальная (≤1 мес)","долгая (≤2 мес)","очень долгая"]
)
df_deals[["deal_id","duration_days","status"]]

In [ ]:
# Работа с NaT (пропуск в datetime)
df_null = pd.DataFrame({
    "start": pd.to_datetime(["2026-01-01","2026-02-01", None, "2026-04-01"]),
    "end":   pd.to_datetime(["2026-01-31", None,"2026-03-15","2026-04-30"]),
})
df_null["duration"] = df_null["end"] - df_null["start"]
# NaT + что угодно = NaT
print(df_null)

### 23.6 DateOffset — бизнес-календарь и умные сдвиги

`DateOffset` — умный сдвиг дат по **бизнес-логике**. В отличие от `Timedelta` (фиксированное число секунд), `DateOffset` учитывает длину месяца, квартала, года.

### Timedelta vs DateOffset

```python
date = pd.Timestamp("2026-01-31")

date + pd.Timedelta(days=30)    # 2026-03-02  (просто +30 дней)
date + pd.DateOffset(months=1)  # 2026-02-28  (следующий месяц)
```

### Часто используемые офсеты

| Класс | Описание |
|-------|---------|
| `pd.DateOffset(days=n)` | +N дней |
| `pd.DateOffset(months=n)` | +N месяцев (умно) |
| `pd.DateOffset(years=n)` | +N лет |
| `pd.offsets.MonthEnd(n)` | n-ый конец месяца |
| `pd.offsets.MonthBegin(n)` | n-ое начало месяца |
| `pd.offsets.QuarterEnd(n)` | n-ый конец квартала |
| `pd.offsets.YearEnd(n)` | n-ый конец года |
| `pd.offsets.BDay(n)` | N рабочих дней |


In [ ]:
# DateOffset vs Timedelta
dates = pd.Series(pd.to_datetime(["2026-01-31","2026-02-28","2026-03-31"]))

print("Исходные даты:")
print(dates.values)

print("\n+ Timedelta(30 дней):")
print((dates + pd.Timedelta(days=30)).values)

print("\n+ DateOffset(months=1):")
print((dates + pd.DateOffset(months=1)).values)

In [ ]:
# Конец и начало периодов
ts = pd.Timestamp("2026-03-15")

print(f"Конец текущего месяца:    {ts + pd.offsets.MonthEnd(0)}")
print(f"Конец следующего месяца:  {ts + pd.offsets.MonthEnd(1)}")
print(f"Начало текущего месяца:   {ts - pd.offsets.MonthBegin(0)}")
print(f"Начало следующего месяца: {ts + pd.offsets.MonthBegin(1)}")
print(f"Конец квартала:           {ts + pd.offsets.QuarterEnd(0)}")
print(f"Конец года:               {ts + pd.offsets.YearEnd(0)}")

In [ ]:
# Рабочие дни (BDay = Business Day)
ts = pd.Timestamp("2026-01-30")  # пятница

print(f"Следующий рабочий день:    {ts + pd.offsets.BDay(1)}")  # понедельник
print(f"Через 5 рабочих дней:      {ts + pd.offsets.BDay(5)}")
print(f"Предыдущий рабочий день:   {ts - pd.offsets.BDay(1)}")

In [ ]:
# Генерация ряда только рабочих дней
biz_days = pd.date_range("2026-01-01", "2026-01-31", freq="B")
print(f"Рабочих дней в январе 2026: {len(biz_days)}")
print(biz_days)

In [ ]:
# Применение к колонке
dates = pd.date_range("2026-01-01", periods=6, freq="MS")
df_off = pd.DataFrame({"month_start": dates})

df_off["month_end"]    = df_off["month_start"] + pd.offsets.MonthEnd(0)
df_off["next_biz_day"] = df_off["month_end"] + pd.offsets.BDay(1)
df_off["quarter_end"]  = df_off["month_start"] + pd.offsets.QuarterEnd(0)

df_off

### 23.7 Period и PeriodIndex

`Period` представляет **временно́й период** (промежуток), а не точку. Например, «январь 2026» — это не конкретный момент, а весь месяц.

### Timestamp vs Period

```
Timestamp("2026-01-15")     → точка на оси времени
Period("2026-01", freq="M") → отрезок [2026-01-01 ... 2026-01-31]
```

### Когда использовать Period

- Отчётные периоды: «выручка за январь», «сделки за Q1»
- Когда важен период, а не конкретная дата
- Удобная агрегация и сравнение: `p1 < p2`, `p2 - p1` (в периодах)


In [ ]:
# Создание
p_month   = pd.Period("2026-01", freq="M")
p_quarter = pd.Period("2026Q1",  freq="Q")
p_year    = pd.Period("2026",    freq="Y")
p_day     = pd.Period("2026-01-15", freq="D")

print(f"Месяц:   {p_month}  | начало: {p_month.start_time} | конец: {p_month.end_time}")
print(f"Квартал: {p_quarter} | начало: {p_quarter.start_time.date()} | конец: {p_quarter.end_time.date()}")
print(f"Год:     {p_year}   | начало: {p_year.start_time.date()} | конец: {p_year.end_time.date()}")
print(f"День:    {p_day}")

In [ ]:
# Арифметика Period
p = pd.Period("2026-03", freq="M")
print(f"Текущий:           {p}")
print(f"Следующий месяц:   {p + 1}")
print(f"Предыдущий месяц:  {p - 1}")
print(f"Через 6 месяцев:   {p + 6}")

p2 = pd.Period("2026-08", freq="M")
print(f"\nРазница: {p2 - p} месяцев")

In [ ]:
# Period в DataFrame — для агрегации
np.random.seed(5)
df_p = pd.DataFrame({
    "date":    pd.date_range("2025-10-01", periods=24, freq="ME"),
    "revenue": np.random.randint(500_000, 2_000_000, 24),
})

df_p["month"]   = df_p["date"].dt.to_period("M")
df_p["quarter"] = df_p["date"].dt.to_period("Q")
df_p["year"]    = df_p["date"].dt.to_period("Y")

print("Выручка по кварталам:")
print(df_p.groupby("quarter")["revenue"].sum())

In [ ]:
# PeriodIndex — массив периодов как индекс
pi = pd.period_range("2025-Q1", periods=8, freq="Q")
df_qi = pd.DataFrame(
    {"revenue": np.random.randint(1_000_000, 5_000_000, 8)},
    index=pi
)
print(df_qi)
print(f"\nQ3 2026: {df_qi.loc['2026Q3', 'revenue']:,}")

### 23.8 Временны́е зоны (Timezone)

Timezone (часовой пояс) — это часто игнорируемая, но важная тема. Ошибки с временными зонами могут незаметно сдвигать данные на несколько часов, что ломает агрегации и сравнения.

### Термины

| Термин | Значение |
|--------|---------|
| **naive** datetime | без информации о часовом поясе |
| **aware** datetime | с указанием часового пояса |
| **UTC** | Coordinated Universal Time — эталонное время без сдвига |
| **tz_localize** | присвоить часовой пояс naive-дате |
| **tz_convert** | конвертировать aware-дату в другой пояс |

### Стратегия работы с timezone

1. **Хранить всё в UTC** — стандарт в production-системах
2. **Конвертировать при отображении** — перевести в локальный пояс только в отчёте
3. **Никогда не смешивать** naive и aware — это ошибка


In [ ]:
# Naive — без часового пояса
naive = pd.Timestamp("2026-03-15 14:30:00")
print(f"Naive: {naive}  | tz: {naive.tz}")

In [ ]:
# Локализация — присвоить часовой пояс
aware_utc = naive.tz_localize("UTC")
aware_msk = naive.tz_localize("Europe/Moscow")

print(f"UTC:    {aware_utc}   | tz: {aware_utc.tz}")
print(f"Moscow: {aware_msk} | tz: {aware_msk.tz}")

In [ ]:
# Конвертация между поясами
aware_utc = pd.Timestamp("2026-03-15 11:00:00", tz="UTC")

print(f"UTC:      {aware_utc}")
print(f"Москва:   {aware_utc.tz_convert('Europe/Moscow')}")
print(f"Нью-Йорк: {aware_utc.tz_convert('America/New_York')}")
print(f"Токио:    {aware_utc.tz_convert('Asia/Tokyo')}")

In [ ]:
# Работа с timezone в DataFrame
df_tz = pd.DataFrame({
    "event_time": pd.date_range("2026-03-15 08:00:00", periods=6, freq="2h"),
    "value":      np.random.randint(100, 1000, 6),
})

# Локализовать в UTC
df_tz["event_time_utc"] = df_tz["event_time"].dt.tz_localize("UTC")

# Конвертировать в московское время
df_tz["event_time_msk"] = df_tz["event_time_utc"].dt.tz_convert("Europe/Moscow")

df_tz[["event_time","event_time_utc","event_time_msk"]]

In [ ]:
# Снять timezone (aware -> naive)
aware = pd.Series(pd.date_range("2026-01-01", periods=3, freq="D", tz="Europe/Moscow"))
print("Aware:", aware.dtype)

# tz_localize(None) убирает timezone
naive = aware.dt.tz_localize(None)
print("Naive:", naive.dtype)

### 23.9 Фильтрация и сортировка по датам

Когда колонка имеет тип `datetime64`, с ней работают как с числами: можно сравнивать, использовать `between`, `query`, фильтровать по компонентам через `.dt`.


In [ ]:
# Тестовый DataFrame
np.random.seed(15)
df_f = pd.DataFrame({
    "date":    pd.date_range("2026-01-01", periods=20, freq="D"),
    "revenue": np.random.randint(100_000, 500_000, 20)
})

# Фильтрация по диапазону
df_f[df_f["date"] >= "2026-01-10"]

In [ ]:
# between
df_f[df_f["date"].between("2026-01-05","2026-01-15")]

In [ ]:
# По компонентам — через .dt
df_f[df_f["date"].dt.weekday < 5]              # только будни

In [ ]:
df_f[df_f["date"].dt.day <= 10]                # первая декада месяца

In [ ]:
df_f[df_f["date"].dt.is_month_end]             # только концы месяцев

In [ ]:
# query — со ссылкой на колонку даты
date_min = pd.Timestamp("2026-01-08")
df_f.query("date >= @date_min and revenue > 200_000")

---
## 24. Временны́е ряды и resample

Временно́й ряд — данные, упорядоченные по времени. pandas предоставляет специальные инструменты: `DatetimeIndex`, `resample()`, скользящие окна.

### resample vs groupby

| | resample | groupby |
|-|----------|---------|
| Ось | только datetime | любые колонки |
| Период | задаётся алиасом `"ME"`, `"D"` | задаётся значениями |
| Пропущенные периоды | создаёт NaN для пустых | не создаёт |
| Временны́е зоны | поддерживает | не поддерживает |

### Требования к resample

- DataFrame должен иметь `DatetimeIndex` (или указать `on="col"`)
- Данные лучше отсортировать по дате

### Ключевые операции с временны́ми рядами

| Операция | Метод | Описание |
|---------|-------|---------|
| Группировка по периоду | `resample()` | суммировать по месяцам, кварталам |
| Скользящее среднее | `rolling()` | сгладить шум |
| Накопительный итог | `expanding()` | нарастающий расчёт |
| Сдвиг | `shift()` | сравнение с предыдущим периодом |
| Темп роста | `pct_change()` | % изменение к предыдущему |


In [ ]:
# Создадим ежедневный ряд с трендом
np.random.seed(7)
n = 90
df_ts = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=n, freq="D"),
    "revenue": (
        np.random.randint(80_000, 200_000, n)
        + np.linspace(0, 60_000, n).astype(int)
        + (np.sin(np.arange(n) * 2*np.pi/7) * 20_000).astype(int)
    ),
    "orders":  np.random.randint(5, 50, n),
    "region":  np.random.choice(["Москва","СПб","Казань"], n),
}).set_index("date")
df_ts.head()

### 24.1 Базовые агрегации

In [ ]:
monthly_sum  = df_ts["revenue"].resample("ME").sum()
monthly_mean = df_ts["revenue"].resample("ME").mean()
weekly_max   = df_ts["revenue"].resample("W").max()
quarterly    = df_ts["revenue"].resample("QE").sum()

print("Выручка по месяцам:")
print(monthly_sum)

### 24.2 Несколько агрегатов и колонок

In [ ]:
monthly_multi = df_ts["revenue"].resample("ME").agg(
    revenue_sum  ="sum",
    revenue_mean ="mean",
    revenue_max  ="max",
    revenue_min  ="min",
).reset_index()
monthly_multi["revenue_mean"] = monthly_multi["revenue_mean"].round(0)
monthly_multi

In [ ]:
# Named aggregation — несколько колонок
monthly_all = df_ts.resample("ME").agg(
    revenue_sum  = ("revenue", "sum"),
    revenue_mean = ("revenue", "mean"),
    orders_total = ("orders",  "sum"),
    orders_mean  = ("orders",  "mean"),
).reset_index()
monthly_all.round(0)

### 24.3 resample с on= — без DatetimeIndex

In [ ]:
df_no_idx = df_ts.reset_index()  # date снова как колонка
monthly = df_no_idx.resample("ME", on="date")["revenue"].sum().reset_index()
monthly

### 24.4 Заполнение пропусков при resample

In [ ]:
# За некоторые периоды может не быть данных — resample создаёт NaN
ts_sparse = df_ts["revenue"].resample("10D").sum()
print("Пропусков:", ts_sparse.isna().sum())

# Заполнить нулём
ts_filled = ts_sparse.fillna(0)
print("После fillna(0):", ts_filled.isna().sum())

### 24.5 OHLC (Open-High-Low-Close)

Классика финансовых данных — за каждый период первое, максимум, минимум, последнее значение.

In [ ]:
ohlc = df_ts["revenue"].resample("W").ohlc()
ohlc.columns = ["open","high","low","close"]
ohlc.head(5)

### 24.6 shift и pct_change — сдвиги и темпы роста

In [ ]:
# MoM (Month-over-Month) темп роста
monthly_df = df_ts["revenue"].resample("ME").sum().reset_index()
monthly_df.columns = ["date","revenue"]
monthly_df["prev_month"]   = monthly_df["revenue"].shift(1)
monthly_df["mom_growth"]   = (monthly_df["revenue"] / monthly_df["prev_month"] - 1).round(4)
monthly_df["mom_growth_%"] = (monthly_df["mom_growth"] * 100).round(1)
monthly_df

In [ ]:
# Сдвиги в исходном ежедневном ряду
df_ts["pct_change"] = df_ts["revenue"].pct_change()    # % изменение к предыдущему
df_ts["prev_day"]   = df_ts["revenue"].shift(1)        # вчера
df_ts["next_day"]   = df_ts["revenue"].shift(-1)       # завтра
df_ts["rev_7d_ago"] = df_ts["revenue"].shift(7)        # неделю назад

df_ts[["revenue","prev_day","pct_change","rev_7d_ago"]].dropna().head(6)

### 24.7 Практические задачи с временны́ми рядами

Реальные сценарии, которые встречаются в аналитической практике.


**Анализ продаж по дням недели и часам**

In [ ]:
np.random.seed(42)
n = 500
df_sales = pd.DataFrame({
    "ts":     pd.date_range("2026-01-01 08:00", periods=n, freq="73min"),
    "amount": np.abs(np.random.normal(15_000, 5_000, n)).astype(int),
    "channel":np.random.choice(["online","offline"], n),
})

df_sales["weekday"]    = df_sales["ts"].dt.day_name()
df_sales["hour"]       = df_sales["ts"].dt.hour
df_sales["is_weekend"] = df_sales["ts"].dt.weekday >= 5

# Средний чек по дням недели
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
by_weekday = (
    df_sales.groupby("weekday")["amount"]
    .agg(avg_amount="mean", orders="count")
    .reindex(weekday_order)
    .reset_index()
)
by_weekday["avg_amount"] = by_weekday["avg_amount"].round(0)
by_weekday

In [ ]:
# Тепловая карта: час × день недели
heatmap_data = df_sales.pivot_table(
    values="amount", index="hour", columns="weekday",
    aggfunc="mean", fill_value=0
)[weekday_order]
heatmap_data.round(0).head(8)

**Возраст клиента и срок договора**

In [ ]:
df_clients = pd.DataFrame({
    "client_id":      [1, 2, 3, 4, 5],
    "birth_date":     pd.to_datetime(["1985-03-15","1990-07-22","2000-01-01",
                                       "1975-11-30","1995-05-10"]),
    "contract_start": pd.to_datetime(["2023-01-01","2022-06-15","2024-03-01",
                                       "2021-09-01","2023-11-20"]),
    "contract_end":   pd.to_datetime(["2027-01-01","2025-06-15","2026-03-01",
                                       "2026-09-01","2025-11-20"]),
})

today = pd.Timestamp.today().normalize()

df_clients["age_years"]      = ((today - df_clients["birth_date"]).dt.days / 365.25).astype(int)
df_clients["contract_months"]= ((df_clients["contract_end"] - df_clients["contract_start"])
                                .dt.days / 30.44).round(0).astype(int)
df_clients["days_until_end"] = (df_clients["contract_end"] - today).dt.days

df_clients["status"] = np.select(
    [df_clients["days_until_end"] < 0, df_clients["days_until_end"] <= 90],
    ["истёк", "скоро истекает"],
    default="активный"
)

df_clients[["client_id","age_years","contract_months","days_until_end","status"]]

**Заполнение пропущенных дат**

In [ ]:
# Часто: данные только по рабочим дням, нужен полный ряд
df_sparse = pd.DataFrame({
    "date":    pd.to_datetime(["2026-01-05","2026-01-06","2026-01-08",
                                "2026-01-12","2026-01-13","2026-01-15"]),
    "revenue": [100_000, 150_000, 120_000, 200_000, 80_000, 170_000],
})

# Создать полный ряд дат и reindex
full_range = pd.date_range(df_sparse["date"].min(), df_sparse["date"].max(), freq="D")
df_full = df_sparse.set_index("date").reindex(full_range).reset_index()
df_full.columns = ["date","revenue"]

df_full["revenue_zero"] = df_full["revenue"].fillna(0)       # выходные = 0
df_full["revenue_ffil"] = df_full["revenue"].ffill()         # предыдущее значение
df_full["is_weekend"]   = df_full["date"].dt.weekday >= 5

df_full

**Дедлайны и бизнес-даты**

In [ ]:
df_tasks = pd.DataFrame({
    "task":         ["Отчёт А","Презентация Б","Аудит В","Согласование Г","Запуск Д"],
    "created_date": pd.to_datetime(["2026-01-05","2026-01-10","2026-01-15",
                                     "2026-01-20","2026-01-25"]),
    "sla_days":     [5, 3, 10, 7, 14],   # рабочих дней
})

# Дедлайн = created_date + sla рабочих дней
df_tasks["deadline"] = df_tasks.apply(
    lambda r: r["created_date"] + pd.offsets.BDay(r["sla_days"]),
    axis=1
)

today = pd.Timestamp("2026-01-28")
df_tasks["days_left"]  = (df_tasks["deadline"] - today).dt.days
df_tasks["is_overdue"] = df_tasks["deadline"] < today

df_tasks["urgency"] = np.select(
    [
        df_tasks["is_overdue"],
        df_tasks["days_left"] <= 2,
        df_tasks["days_left"] <= 5,
    ],
    ["🔴 Просрочено", "🟠 Срочно", "🟡 Скоро"],
    default="🟢 В норме"
)

df_tasks[["task","created_date","deadline","days_left","urgency"]]

### 24.8 Шпаргалка по датам

```python
# СОЗДАНИЕ И ПАРСИНГ
pd.Timestamp("2026-03-15")
pd.Timestamp.now() / pd.Timestamp.today()
pd.to_datetime(series, errors="coerce")
pd.to_datetime(series, format="%d.%m.%Y")
pd.to_datetime(series, unit="s")               # из Unix timestamp
pd.to_datetime(df[["year","month","day"]])

pd.date_range("2026-01-01", periods=12, freq="ME")
pd.date_range("2026-01-01", "2026-12-31", freq="B")  # рабочие дни
pd.period_range("2026-01", periods=6, freq="M")

# АКСЕССОР .dt
.dt.year / .month / .day / .quarter / .day_of_year / .day_of_week
.dt.hour / .minute / .second
.dt.days_in_month
.dt.day_name() / .month_name()
.dt.isocalendar()                              # year, week, day
.dt.is_month_start / .is_month_end
.dt.is_quarter_start / .is_quarter_end
.dt.is_year_start / .is_year_end
.dt.is_leap_year
.dt.strftime("%Y-%m-%d")
.dt.to_period("M") / .dt.to_period("Q")
.dt.floor("h") / .dt.ceil("h") / .dt.round("15min")
.dt.date / .dt.time

# TIMEDELTA
pd.Timedelta("3 days 4 hours")
pd.Timedelta(days=5, hours=3)
(date_end - date_start).dt.days
(date_end - date_start).dt.total_seconds()
.dt.components

# DATEOFFSET
ts + pd.DateOffset(months=1)                   # умный сдвиг
ts + pd.offsets.MonthEnd(0)                    # конец текущего месяца
ts + pd.offsets.MonthBegin(1)                  # начало следующего месяца
ts + pd.offsets.BDay(5)                        # +5 рабочих дней
ts + pd.offsets.QuarterEnd(0)
pd.date_range(..., freq="B")                   # только рабочие дни

# PERIOD
pd.Period("2026-01", freq="M")
p.start_time / p.end_time
p + 1 / p - 1
p2 - p1                                        # разность в периодах

# TIMEZONE
series.dt.tz_localize("UTC")                   # naive -> aware
series.dt.tz_convert("Europe/Moscow")         # aware -> другой пояс
series.dt.tz_localize(None)                   # снять timezone

# RESAMPLE
df.set_index("date").resample("ME").sum()
df.resample("W", on="date").agg(
    rev=("revenue","sum"), cnt=("orders","count")
).reset_index()
.resample("W").ohlc()                          # open-high-low-close

# ФИЛЬТРАЦИЯ ПО ДАТАМ
df[df["date"] >= "2026-01-01"]
df[df["date"].between("2026-01-01","2026-03-31")]
df[df["date"].dt.month == 3]
df[df["date"].dt.weekday < 5]                  # рабочие дни
df[df["date"].dt.is_month_end]

# ЗАПОЛНЕНИЕ ПРОПУЩЕННЫХ ДАТ
full = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
df.set_index("date").reindex(full).reset_index()
```


---
## 25. Оконные функции

Оконные функции вычисляют значения для каждой строки с учётом **окна соседних строк**. В отличие от агрегации, DataFrame не сжимается.

### Виды оконных функций

| Тип | Метод | Описание |
|-----|-------|---------|
| **Rolling** | `.rolling(n)` | Фиксированное окно из N строк |
| **Expanding** | `.expanding()` | Окно от начала ряда до текущей строки |
| **EWM** | `.ewm(span=n)` | Экспоненциальное взвешивание (новые данные важнее) |

### Когда применять
- **Rolling mean** — сгладить зашумлённые данные
- **Expanding sum** — нарастающий итог бюджета
- **EWM** — прогнозирование, где свежие данные важнее старых


In [ ]:
df_ts["roll_7d_mean"] = df_ts["revenue"].rolling(window=7).mean()
df_ts["roll_7d_max"]  = df_ts["revenue"].rolling(window=7).max()
df_ts["roll_7d_min"]  = df_ts["revenue"].rolling(window=7).min()
# center=True — симметричное окно вокруг текущей точки
df_ts["roll_7d_ctr"]  = df_ts["revenue"].rolling(window=7, center=True).mean()

df_ts[["revenue","roll_7d_mean","roll_7d_max","roll_7d_min"]].dropna().head(8)

In [ ]:
df_ts["exp_mean"] = df_ts["revenue"].expanding().mean()
df_ts["exp_sum"]  = df_ts["revenue"].expanding().sum()
df_ts[["revenue","exp_mean","exp_sum"]].head(8)

In [ ]:
# span=7: примерно 7 дней "памяти"
df_ts["ewm_7"]  = df_ts["revenue"].ewm(span=7).mean()
df_ts["ewm_30"] = df_ts["revenue"].ewm(span=30).mean()
df_ts[["revenue","roll_7d_mean","ewm_7","ewm_30"]].dropna().head(8)

### 25.4 min_periods — допустить неполное окно

По умолчанию `rolling(window=n)` возвращает `NaN` для первых `n-1` строк (окно не заполнено). С параметром `min_periods=k` окно начинает работать уже с `k` точек.

Это полезно, чтобы **не терять начало ряда** — особенно в коротких сериях.


In [ ]:
s = pd.Series([10, 20, 30, 40, 50, 60, 70])

# По умолчанию: первые 6 строк -> NaN
print("rolling(7).mean():")
print(s.rolling(7).mean().tolist())

# С min_periods=1 — расчёт начинается сразу
print("\nrolling(7, min_periods=1).mean():")
print(s.rolling(7, min_periods=1).mean().round(1).tolist())

### 25.5 Окно на основе даты (offset window)

`rolling()` принимает не только число строк, но и **временной интервал** — например, `"7D"` (7 дней). Это особенно важно для рядов с пропусками: оконо считается по календарным дням, а не по количеству записей.


In [ ]:
# Ряд с пропусками — некоторых дней нет
df = pd.DataFrame({
    "date":    pd.to_datetime(["2026-01-01","2026-01-02","2026-01-05",
                                "2026-01-07","2026-01-10","2026-01-15"]),
    "revenue": [100, 120, 150, 200, 180, 220],
}).set_index("date")

# rolling по 7 КАЛЕНДАРНЫХ дням (а не 7 записям)
df["roll_7D"] = df["revenue"].rolling("7D").mean()
df.round(1)

### 25.6 Произвольная функция в окне через .apply()

`rolling().apply()` позволяет применить любую функцию к окну. Полезно для нестандартных метрик: размах, коэффициент вариации, кастомный score.


In [ ]:
s = pd.Series([10, 20, 15, 25, 30, 22, 35])

# Размах (max - min) в окне 3
def range_func(x):
    return x.max() - x.min()

s.rolling(3).apply(range_func)

---
## 26. Ранжирование и накопительные расчёты

### rank() — ранжирование

`rank()` присваивает каждому значению ранг. Параметр `method` задаёт поведение при одинаковых значениях:

| method | Описание |
|--------|---------|
| `"average"` | средний ранг (умолчание) |
| `"min"` | наименьший из одинаковых |
| `"dense"` | без пропусков: 1, 2, 3 (никогда 1, 3) |
| `"first"` | в порядке появления |


In [ ]:
df_r = df[["client_id","manager","region","revenue","margin"]].copy()

df_r["revenue_rank"] = df_r["revenue"].rank(ascending=False, method="dense").astype(int)
df_r["region_rank"]  = df_r.groupby("region")["revenue"].rank(
    ascending=False, method="dense"
).astype(int)

df_r.sort_values("revenue_rank")[
    ["client_id","manager","region","revenue","revenue_rank","region_rank"]
].head(8)

In [ ]:
# Принцип Парето — сколько клиентов дают 80% выручки
df_s = df_r.sort_values("revenue", ascending=False).copy()
df_s["rev_cumsum"]  = df_s["revenue"].cumsum()
df_s["rev_cumpct"]  = (df_s["rev_cumsum"] / df_s["revenue"].sum() * 100).round(1)

pareto = df_s[df_s["rev_cumpct"] <= 80]
print(f"Правило Парето: {len(pareto)} из {len(df_s)} клиентов дают 80% выручки")
df_s[["client_id","revenue","rev_cumsum","rev_cumpct"]].head(8)

In [ ]:
# cumcount и cumsum внутри группы
df_r_s = df_r.sort_values(["manager","revenue"], ascending=[True,False])

df_r_s["deal_rank_in_mgr"]  = df_r_s.groupby("manager").cumcount() + 1
df_r_s["mgr_running_rev"]   = df_r_s.groupby("manager")["revenue"].cumsum()

df_r_s[["manager","revenue","deal_rank_in_mgr","mgr_running_rev"]].head(10)

---
## 27. MultiIndex (составной индекс)

MultiIndex — индекс из **нескольких уровней**. Автоматически создаётся при `groupby` по нескольким колонкам без `reset_index()`.

### Практическая рекомендация

В большинстве задач MultiIndex усложняет работу. Почти всегда лучше **сразу сбрасывать** через `reset_index()` и работать с плоской таблицей. Исключение — `DatetimeIndex` для временных рядов.


In [ ]:
result_mi = df.groupby(["manager","region"])["revenue"].agg(["sum","count","mean"])
print("Тип индекса:", type(result_mi.index))
result_mi

In [ ]:
# Сразу сбрасывать — лучшая практика
result_flat = (
    df.groupby(["manager","region"])["revenue"]
    .agg(revenue_sum="sum", deals="count", avg_revenue="mean")
    .reset_index()
)
result_flat

In [ ]:
# MultiIndex в колонках после pivot_table
pivot_mi = pd.pivot_table(df, values=["revenue","margin"], index="manager",
                          columns="region", aggfunc="sum", fill_value=0)
print("Колонки MultiIndex:", pivot_mi.columns.tolist()[:4])

# Выровнять в одноуровневые колонки
pivot_mi.columns = ["_".join(col) for col in pivot_mi.columns]
pivot_mi.head()

### 27.3 Когда MultiIndex реально полезен

Несмотря на сложность, MultiIndex имеет нишу:

1. **Иерархические отчёты** — когда у вас «континент → страна → город»
2. **Сводные таблицы с MultiIndex колонок** — после `pivot_table` с несколькими `values`
3. **Финансовые временные ряды** — `(дата, инструмент)` как ключ

### Извлечение из MultiIndex — .xs() и .loc[]


In [ ]:
# Создадим многоуровневый индекс
arrays = [
    ["Europe","Europe","Europe","Asia","Asia"],
    ["FR","DE","IT","CN","JP"],
]
df_mi = pd.DataFrame(
    {"population": [67, 83, 60, 1400, 125], "gdp": [2.9, 4.2, 2.1, 17.0, 4.9]},
    index=pd.MultiIndex.from_arrays(arrays, names=["region","country"])
)
print(df_mi)

In [ ]:
# Все страны Европы через .xs (cross-section)
df_mi.xs("Europe", level="region")

In [ ]:
# Сглаживание MultiIndex обратно в плоскую таблицу
df_flat = df_mi.reset_index()
df_flat

---
## 28. Категориальные данные (Categorical)

`category` — специальный тип pandas для колонок с **ограниченным набором повторяющихся значений**.

### Как хранится category

```
object:   ["Москва","СПб","Казань","Москва","Москва"]  → 5 строк × ~50 байт = 250 байт
category: [1, 2, 0, 1, 1] + {0:"Казань", 1:"Москва", 2:"СПб"} → ~25 байт
```

На миллионе строк с 10 уникальными значениями — экономия в **десятки раз**.

Дополнительная возможность: **упорядоченные категории** — можно сравнивать `"low" < "medium" < "high"`.


In [ ]:
df["region"]  = df["region"].astype("category")
df["manager"] = df["manager"].astype("category")

print("Категории region:", df["region"].cat.categories.tolist())
print("Коды (первые 5):",  df["region"].cat.codes.head().tolist())

In [ ]:
df_prio = pd.DataFrame({
    "task":     ["Звонок А","Встреча Б","Отчёт В","Презентация Г","Звонок Д"],
    "priority": ["high","low","medium","high","low"],
    "hours":    [2, 8, 4, 6, 1]
})

df_prio["priority"] = pd.Categorical(
    df_prio["priority"],
    categories=["low","medium","high"],
    ordered=True
)

print(df_prio.sort_values("priority"))
print()
print("Задачи с приоритетом >= medium:")
print(df_prio[df_prio["priority"] >= "medium"])

---
## 29. Цепочки методов: assign и pipe

### Зачем нужны цепочки?

Стандартный код с промежуточными переменными:
```python
df1 = df.copy()
df1["margin"] = df1["revenue"] - df1["cost"]
df1 = df1[df1["margin"] > 0]
df1 = df1.sort_values("margin", ascending=False)
```

С цепочками методов (Method Chaining):
```python
df1 = (
    df
    .assign(margin = lambda d: d["revenue"] - d["cost"])
    .query("margin > 0")
    .sort_values("margin", ascending=False)
)
```

**Преимущества:**
- нет промежуточных переменных `df1`, `df2`, `df3`
- код читается сверху вниз как инструкция
- легко добавить или убрать шаг

### 29.1 assign — добавить колонки в цепочке


In [ ]:
result = (
    df
    .assign(
        margin       = lambda d: d["revenue"] - d["cost"],
        margin_rate  = lambda d: (d["margin"] / d["revenue"]).round(4),
        is_vip       = lambda d: d["revenue"] >= 1_200_000,
        size_label   = lambda d: np.where(d["revenue"] >= 1_000_000, "large", "small"),
    )
    .query("margin > 0")
    .sort_values("margin_rate", ascending=False)
    [["manager","region","revenue","margin_rate","is_vip","size_label"]]
    .head(8)
)
result

### 29.2 pipe — применить произвольную функцию в цепочке

`pipe(func)` передаёт весь DataFrame в функцию. Позволяет вставить в цепочку любую операцию.


In [ ]:
def log_shape(df, label=""):
    print(f"{label}: {df.shape[0]} строк × {df.shape[1]} колонок")
    return df

def normalize_strings(df, cols):
    df = df.copy()
    for col in cols:
        df[col] = df[col].str.strip().str.title()
    return df

def add_tier(df, low=400_000, high=1_000_000):
    df = df.copy()
    df["tier"] = np.select(
        [df["revenue"] >= high, df["revenue"] >= low],
        ["Premium", "Standard"], default="Basic"
    )
    return df

result_pipe = (
    df
    .pipe(log_shape, label="Исходный")
    .pipe(normalize_strings, cols=["manager","region"])
    .pipe(add_tier, low=400_000, high=1_000_000)
    .query("margin > 0")
    .pipe(log_shape, label="После фильтрации")
    [["manager","region","revenue","tier"]]
)
result_pipe.head()

---
## 30. explode — работа со списками в ячейках

Иногда в ячейке DataFrame хранится **список** значений (теги, категории, позиции заказа). Метод `explode()` разворачивает список — каждое значение становится отдельной строкой.

**Типичные источники:**
- JSON-поля из API (массив тегов, список SKU)
- данные после `str.split()` (строка с запятыми -> список)


In [ ]:
df_products = pd.DataFrame({
    "product_id": [1, 2, 3, 4],
    "name":       ["Ноутбук Pro","Телефон X","Планшет Air","Наушники Z"],
    "tags":       [["sale","new","popular"],["sale"],["premium","new"],["popular","sale","premium"]],
    "price":      [89_000, 45_000, 62_000, 15_000]
})
df_products

In [ ]:
df_exploded = df_products.explode("tags").reset_index(drop=True)
df_exploded

In [ ]:
print("Кол-во товаров по тегам:")
print(df_exploded.groupby("tags")["product_id"].count().sort_values(ascending=False))
print()
print("Средняя цена по тегам:")
print(df_exploded.groupby("tags")["price"].mean().sort_values(ascending=False))

In [ ]:
# Частый случай: строка с запятыми -> explode
df_csv_tags = pd.DataFrame({
    "id": [1, 2, 3],
    "tags": ["sale,new,popular", "sale", "premium,new"],
})
df_csv_tags["tags_list"] = df_csv_tags["tags"].str.split(",")
df_csv_tags.explode("tags_list").reset_index(drop=True)

---
## 31. Работа с данными как с NumPy

pandas и NumPy глубоко интегрированы. Иногда нужен чистый NumPy-массив — для ML-моделей, матричных операций или библиотек, не принимающих DataFrame.


In [ ]:
arr_2d = df[["revenue","cost","margin"]].to_numpy()
print("Тип:", type(arr_2d))
print("Форма:", arr_2d.shape)
print("dtype:", arr_2d.dtype)
arr_2d[:3]

In [ ]:
arr_1d = df["revenue"].to_numpy()
print(arr_1d[:5])

In [ ]:
# Корреляционная матрица через NumPy
X = df[["revenue","cost","margin","orders"]].to_numpy()
corr = np.corrcoef(X.T)
pd.DataFrame(corr,
             index=["revenue","cost","margin","orders"],
             columns=["revenue","cost","margin","orders"]).round(3)

In [ ]:
# Стандартизация (z-score) — типичная предобработка для ML
X = df[["revenue","cost"]].to_numpy().astype(float)
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
print("Среднее после нормализации:", X_norm.mean(axis=0).round(10))
print("Ст.откл. после нормализации:", X_norm.std(axis=0).round(10))

---
## 32. Производительность и память

С ростом данных неоптимальный код начинает падать с `MemoryError` или работает минуты вместо секунд.

### Иерархия оптимизаций (от самой эффективной)

1. **Читать только нужные колонки** (`usecols`) — не грузить лишнее
2. **Правильные типы при чтении** (`dtype`, `category`) — не конвертировать потом
3. **Downcast числовых колонок** (`int64` → `int16`, `float64` → `float32`)
4. **category для строк** с малым кол-вом уникальных значений
5. **Parquet вместо CSV** для хранения и повторного чтения
6. **Chunking** для файлов больше оперативной памяти

### Порядок работы с памятью: measure → optimize → measure again


In [ ]:
memory_by_col = df.memory_usage(deep=True).sort_values(ascending=False)
print("Память по колонкам (байт):")
print(memory_by_col)
print(f"\nИтого: {memory_by_col.sum() / 1024:.1f} KB")

In [ ]:
df_heavy = df.copy()
df_light = df.copy()

df_light["orders"]  = df_light["orders"].astype("uint8")
df_light["manager"] = df_light["manager"].astype("category")
df_light["region"]  = df_light["region"].astype("category")

before = df_heavy.memory_usage(deep=True).sum()
after  = df_light.memory_usage(deep=True).sum()
print(f"До:    {before:,} байт")
print(f"После: {after:,} байт")
print(f"Экономия: {(1-after/before)*100:.0f}%")

In [ ]:
# downcast — автоматически наименьший подходящий тип
large = pd.Series([1, 5, 100, 255, 10])
print("До:    ", large.dtype)
print("После: ", pd.to_numeric(large, downcast="integer").dtype)   # uint8

floats = pd.Series([1.1, 2.5, 100.0, 3.14])
print("Float до:    ", floats.dtype)
print("Float после: ", pd.to_numeric(floats, downcast="float").dtype)  # float32

### 32.4 query() и eval() — выражения как строки + ускорение

Внутри pandas для парсинга `query("col > 0 and other < 10")` используется библиотека **numexpr**. На больших DataFrame (от ~100 000 строк) это **быстрее** обычной булевой маски, потому что:
- не создаёт промежуточных булевых массивов
- параллелится на несколько ядер CPU

Аналогично, `df.eval("c = a + b")` вычисляет арифметические выражения как строки.


In [ ]:
np.random.seed(0)
n = 200_000
df = pd.DataFrame({
    "a": np.random.randint(0, 1000, n),
    "b": np.random.randint(0, 1000, n),
})

import time

# Обычная булева маска
t0 = time.perf_counter()
m1 = df[(df["a"] > 500) & (df["b"] < 500)]
t_mask = time.perf_counter() - t0

# query
t0 = time.perf_counter()
m2 = df.query("a > 500 and b < 500")
t_query = time.perf_counter() - t0

print(f"Маска:  {t_mask*1000:.1f} мс")
print(f"query:  {t_query*1000:.1f} мс")
print(f"Совпадает: {m1.equals(m2)}")

In [ ]:
# eval — вычисление выражений из строки
df.eval("c = a + b", inplace=True)
df.eval("ratio = a / (b + 1)", inplace=True)
df.head()

### 32.5 Когда переходить на Polars / DuckDB / Spark

Pandas прекрасно работает на данных **до ~10 ГБ в RAM**. За этой границей стоит подумать о других инструментах:

| Размер данных | Инструмент |
|--------------|-----------|
| До 1 ГБ | pandas, без оптимизаций |
| 1–10 ГБ | pandas + Parquet + категории + downcast |
| 10–100 ГБ | **Polars** (Rust, lazy, parallel) или **DuckDB** (SQL, parquet-friendly) |
| > 100 ГБ | **PySpark**, **Dask** (распределённые вычисления) |

Polars и DuckDB имеют **API, похожий на pandas**, и в большинстве случаев их можно использовать как drop-in замену.


---
## 33. Copy-on-Write (CoW)

### Проблема до CoW

До pandas 2.0 поведение при изменении части DataFrame было непредсказуемым: иногда изменялся оригинал, иногда нет — зависело от внутренней реализации. Это приводило к трудноуловимым багам типа `SettingWithCopyWarning`.

### Copy-on-Write (pandas 2.0+ опционально, pandas 3.0+ по умолчанию)

Простое правило:
> **Любая операция, возвращающая новый DataFrame, никогда не изменяет исходный.**

### Практические правила

| Ситуация | Правильный код |
|---------|----------------|
| Изменить значения по условию | `df.loc[cond, "col"] = value` |
| Работать с подмножеством | `sub = df[cond].copy()` |
| Цепочные изменения | использовать `assign()` |


In [ ]:
# ❌ Не работает в CoW — цепочное присваивание теряется
# df[df["revenue"] > 1_000_000]["tier"] = "premium"

# ✅ Правильно — через .loc
df.loc[df["revenue"] > 1_000_000, "tier"] = "premium"
df.loc[df["revenue"] <= 1_000_000, "tier"] = "standard"
print(df["tier"].value_counts())

In [ ]:
# ✅ Работа с копией
df_moscow = df[df["region"] == "Москва"].copy()
df_moscow["local_tax"] = df_moscow["revenue"] * 0.02   # безопасно
df_moscow.head(3)

In [ ]:
# ✅ Цепочки через assign() — всегда безопасно
result = (
    df
    .assign(
        margin_pct = lambda d: (d["margin"] / d["revenue"] * 100).round(1),
        grade      = lambda d: np.where(d["margin_pct"] > 20, "A", "B")
    )
    .query("margin_pct > 0")
)
result[["manager","margin_pct","grade"]].head()

---
## 34. Стилизация таблиц (Styler)

`Styler` позволяет визуально оформить DataFrame в Jupyter Notebook. Удобно для финальных отчётов и презентаций данных без построения графиков.

### Возможности

| Метод | Эффект |
|-------|--------|
| `.format()` | форматирование чисел, дат |
| `.background_gradient()` | тепловая карта по значениям |
| `.highlight_max()` / `.highlight_min()` | подсветить экстремумы |
| `.bar()` | мини-гистограмма в ячейке |
| `.apply()` | произвольная стилизация через функцию |
| `.set_caption()` | заголовок таблицы |


In [ ]:
df_show = (
    df.groupby("manager").agg(
        revenue_sum = ("revenue",     "sum"),
        deals       = ("client_id",   "count"),
        margin_mean = ("margin_rate", "mean"),
    ).reset_index()
    .sort_values("revenue_sum", ascending=False)
)

(
    df_show.style
    .format({
        "revenue_sum": "{:,.0f} ₽",
        "deals":       "{:d}",
        "margin_mean": "{:.1%}",
    })
    .background_gradient(subset=["revenue_sum"], cmap="YlGn")
    .background_gradient(subset=["margin_mean"], cmap="RdYlGn")
    .highlight_max(subset=["revenue_sum"], color="#c6efce")
    .highlight_min(subset=["margin_mean"], color="#ffc7ce")
    .bar(subset=["deals"], color="#d9e8fb")
    .set_caption("📊 Итоговый отчёт по менеджерам")
)

In [ ]:
def highlight_low_margin(row):
    color = "#ffe5e5" if row["margin_mean"] < 0.05 else ""
    return [f"background-color: {color}"] * len(row)

df_show.style.apply(highlight_low_margin, axis=1).format({
    "revenue_sum": "{:,.0f}",
    "margin_mean": "{:.1%}",
})

---
## 35. Частые ошибки

Ошибки, которые встречаются у большинства новичков. Зная их заранее, сэкономите часы отладки.


### 35.1 and / or вместо & / |

Python-операторы `and`/`or` не работают поэлементно с Series — они выбрасывают `ValueError`.

In [ ]:
# ❌ Ошибка
# df[(df["revenue"] > 1_000_000) and (df["margin"] > 0)]   # ValueError!

# ✅ Правильно — побитовые операторы, каждое условие в скобках
result = df[(df["revenue"] > 1_000_000) & (df["margin"] > 0)]
print(f"Строк: {len(result)}")

### 35.2 Забытые скобки вокруг условий

`&` имеет более высокий приоритет, чем `>` и `<`. Без скобок результат непредсказуем.

In [ ]:
# ❌ Ошибка — интерпретируется не так, как ожидается
# df[df["revenue"] > 1_000_000 & df["margin"] > 0]

# ✅ Каждое условие в скобках
result = df[(df["revenue"] > 1_000_000) & (df["margin"] > 0)]
print(result.shape)

### 35.3 Series vs DataFrame

In [ ]:
print(type(df["revenue"]))    # <class 'pandas.core.series.Series'>
print(type(df[["revenue"]]))  # <class 'pandas.core.frame.DataFrame'>
# Важно: df.merge() ожидает DataFrame, а не Series

### 35.4 Деление на ноль → inf или NaN

In [ ]:
df_zero = pd.DataFrame({"revenue":[1000,0,500], "cost":[800,0,200]})
df_zero["margin"]      = df_zero["revenue"] - df_zero["cost"]
df_zero["margin_rate"] = df_zero["margin"] / df_zero["revenue"]   # деление на 0!
print(df_zero)
print()
# ✅ Защита через np.where
df_zero["safe_rate"] = np.where(
    df_zero["revenue"] != 0,
    df_zero["margin"] / df_zero["revenue"],
    np.nan
)
print(df_zero[["revenue","margin","safe_rate"]])

### 35.5 Merge без проверки — потеря или дублирование строк

In [ ]:
# Дубли в правой таблице -> строки дублируются после merge!
df_ords_dup = pd.concat([df_orders, df_orders.iloc[:2]], ignore_index=True)
merged_bad  = df_clients.merge(df_ords_dup, on="client_id", how="left")
print(f"Клиентов: {len(df_clients)}, после merge: {len(merged_bad)}")  # неожиданно больше!

# ✅ Всегда проверять с indicator
merged_ok = df_clients.merge(df_orders, on="client_id", how="left", indicator=True)
print(merged_ok["_merge"].value_counts())

### 35.6 inplace=True — антипаттерн

In [ ]:
# ❌ inplace=True мутирует DataFrame неявно, ломается в CoW-режиме
# df.sort_values("revenue", inplace=True)   # плохо

# ✅ Всегда присваивать явно
df = df.sort_values("revenue", ascending=False)
df.head(3)

### 35.7 iterrows() — катастрофически медленно

In [ ]:
# ❌ iterrows() — каждая строка создаёт объект Series, O(n) на Python
# for idx, row in df.iterrows():
#     df.at[idx, "margin_rate"] = row["margin"] / row["revenue"]

# ✅ Векторная операция — в тысячи раз быстрее
df["margin_rate"] = df["margin"] / df["revenue"]
print("Правило: iterrows() — заменить на векторную операцию или apply")

---
## 36. Бизнес-примеры

### 36.1 План-факт анализ

In [ ]:
df_pf = pd.DataFrame({
    "manager":    ["Иванов","Петров","Сидорова","Козлов","Миронова"],
    "sales_fact": [980_000, 1_250_000, 890_000, 1_100_000, 1_450_000],
    "sales_plan": [1_100_000, 1_000_000, 950_000, 1_050_000, 1_300_000],
})

df_pf["completion"] = (df_pf["sales_fact"] / df_pf["sales_plan"]).round(4)
df_pf["delta"]      = df_pf["sales_fact"] - df_pf["sales_plan"]
df_pf["status"]     = np.where(df_pf["completion"] >= 1, "✅ Выполнен", "❌ Не выполнен")

df_pf.sort_values("completion", ascending=False).style.format({
    "sales_fact": "{:,.0f}", "sales_plan": "{:,.0f}",
    "delta": "{:+,.0f}",    "completion": "{:.1%}",
}).background_gradient(subset=["completion"], cmap="RdYlGn")

### 36.2 P&L (маржинальность)

In [ ]:
np.random.seed(55)
df_pl = pd.DataFrame({
    "client":  ["Альфа","Бета","Гамма","Дельта","Эпсилон","Зета","Эта","Тета","Йота","Каппа"],
    "revenue": np.random.randint(200_000, 2_000_000, 10),
    "cogs":    np.random.randint(100_000, 1_500_000, 10),
    "opex":    np.random.randint(10_000,  100_000,   10),
})

df_pl["gross_profit"]     = df_pl["revenue"] - df_pl["cogs"]
df_pl["gross_margin"]     = df_pl["gross_profit"] / df_pl["revenue"]
df_pl["operating_profit"] = df_pl["gross_profit"] - df_pl["opex"]
df_pl["operating_margin"] = df_pl["operating_profit"] / df_pl["revenue"]

totals = df_pl[["revenue","cogs","opex","gross_profit","operating_profit"]].sum()
print(f"Выручка:              {totals['revenue']:>12,.0f} ₽")
print(f"Валовая прибыль:      {totals['gross_profit']:>12,.0f} ₽  ({totals['gross_profit']/totals['revenue']:.1%})")
print(f"Операционная прибыль: {totals['operating_profit']:>12,.0f} ₽  ({totals['operating_profit']/totals['revenue']:.1%})")

### 36.3 Воронка продаж

In [ ]:
df_funnel = pd.DataFrame({
    "stage":  ["Лиды","Квалификация","КП отправлено","Переговоры","Сделка"],
    "count":  [1000, 650, 380, 210, 120],
})

df_funnel["cr_from_prev"]  = (df_funnel["count"] / df_funnel["count"].shift(1)).round(3)
df_funnel["cr_from_start"] = (df_funnel["count"] / df_funnel["count"].iloc[0]).round(3)
df_funnel["drop"]          = df_funnel["count"] - df_funnel["count"].shift(-1)

df_funnel.style.format({
    "count":          "{:,.0f}",
    "cr_from_prev":   "{:.1%}",
    "cr_from_start":  "{:.1%}",
    "drop":           "{:,.0f}",
})

### 36.4 Рекламные метрики (ROMI, CPC, CPL)

In [ ]:
df_ads = pd.DataFrame({
    "channel":     ["Яндекс.Директ","Google Ads","VK Реклама","TikTok Ads","Телеграм"],
    "ad_spend":    [500_000, 300_000, 150_000, 100_000, 80_000],
    "impressions": [1_500_000, 900_000, 600_000, 2_000_000, 400_000],
    "clicks":      [25_000, 18_000, 12_000, 30_000, 8_000],
    "leads":       [1_200, 900, 450, 600, 200],
    "sales":       [120, 80, 40, 50, 20],
    "revenue":     [3_600_000, 2_400_000, 1_000_000, 1_500_000, 600_000],
})

df_ads["ctr"]      = (df_ads["clicks"] / df_ads["impressions"]).round(4)
df_ads["cpc"]      = (df_ads["ad_spend"] / df_ads["clicks"]).round(2)
df_ads["cpl"]      = (df_ads["ad_spend"] / df_ads["leads"]).round(2)
df_ads["cr_lead"]  = (df_ads["leads"] / df_ads["clicks"]).round(4)
df_ads["cr_sale"]  = (df_ads["sales"] / df_ads["leads"]).round(4)
df_ads["romi"]     = ((df_ads["revenue"] - df_ads["ad_spend"]) / df_ads["ad_spend"]).round(2)

df_ads[["channel","ad_spend","cpc","cpl","cr_lead","cr_sale","romi"]].style.format({
    "ad_spend": "{:,.0f} ₽",
    "cpc":      "{:.1f} ₽",
    "cpl":      "{:.1f} ₽",
    "cr_lead":  "{:.2%}",
    "cr_sale":  "{:.2%}",
    "romi":     "{:.1f}x",
}).background_gradient(subset=["romi"], cmap="RdYlGn")

### 36.5 Дебиторская задолженность — риск-флаги

In [ ]:
df_debt = pd.DataFrame({
    "client":           ["ООО Альфа","ООО Бета","ООО Гамма","ООО Дельта","ООО Зета"],
    "debt_rub":         [2_500_000, 150_000, 1_800_000, 800_000, 50_000],
    "overdue_days":     [95, 10, 70, 45, 3],
    "payments_last_3m": [0, 5, 1, 3, 8],
    "credit_limit":     [1_000_000, 500_000, 2_000_000, 1_000_000, 300_000],
})

df_debt["over_limit"] = df_debt["debt_rub"] > df_debt["credit_limit"]
df_debt["risk_score"] = (
    (df_debt["debt_rub"] > 1_000_000).astype(int)  +
    (df_debt["overdue_days"] > 60).astype(int)      +
    (df_debt["payments_last_3m"] == 0).astype(int)  +
    df_debt["over_limit"].astype(int)
)

df_debt["risk_level"] = np.select(
    [df_debt["risk_score"] >= 3, df_debt["risk_score"] >= 2],
    ["🔴 Критический", "🟡 Высокий"],
    default="🟢 Низкий"
)

df_debt[["client","debt_rub","overdue_days","payments_last_3m","risk_score","risk_level"]]

### 36.6 Когортный анализ (упрощённый)

In [ ]:
np.random.seed(99)
dates = pd.date_range("2025-10-01", "2026-03-31", freq="D")
df_txn = pd.DataFrame({
    "client_id": np.random.choice(range(1, 51), 200),
    "txn_date":  np.random.choice(dates, 200),
    "amount":    np.random.randint(1_000, 50_000, 200),
}).sort_values("txn_date")

# Когорта = месяц первой покупки
first_purchase = df_txn.groupby("client_id")["txn_date"].min().rename("cohort_date")
df_txn = df_txn.merge(first_purchase, on="client_id")

df_txn["cohort_month"] = df_txn["cohort_date"].dt.to_period("M")
df_txn["txn_month"]    = df_txn["txn_date"].dt.to_period("M")
df_txn["months_since"] = (df_txn["txn_month"] - df_txn["cohort_month"]).apply(lambda x: x.n)

cohort_table = df_txn.groupby(["cohort_month","months_since"])["client_id"].nunique().unstack()
print("Когортная таблица (активных клиентов):")
cohort_table

---
## 37. Шаблон анализа и шпаргалка

In [ ]:
# ═══════════════════════════════════════════════
# ШАБЛОН ТИПИЧНОГО PANDAS-АНАЛИЗА
# ═══════════════════════════════════════════════

import numpy as np
import pandas as pd

# 1. ЗАГРУЗКА
# df = pd.read_csv("file.csv", sep=";", encoding="cp1251")

# 2. ПЕРВИЧНЫЙ ОСМОТР
# df.head()
# df.info()
# df.describe()
# df.isna().sum()
# df.duplicated().sum()

# 3. ПРИВЕДЕНИЕ ТИПОВ
# df["date"]    = pd.to_datetime(df["date"], errors="coerce")
# df["amount"]  = pd.to_numeric(df["amount"], errors="coerce")
# df["region"]  = df["region"].astype("category")

# 4. ОЧИСТКА
# df = df.drop_duplicates()
# df["amount"] = df["amount"].fillna(0)

# 5. РАСЧЁТ ПОКАЗАТЕЛЕЙ
# df["margin"]      = df["revenue"] - df["cost"]
# df["margin_rate"] = np.where(df["revenue"]!=0, df["margin"]/df["revenue"], np.nan)

# 6. СЕГМЕНТАЦИЯ
# df["segment"] = np.select(
#     [df["revenue"]>=1_000_000, df["revenue"]>=500_000],
#     ["large", "medium"], default="small"
# )

# 7. ГРУППИРОВКА
# result = df.groupby("manager").agg(
#     revenue_sum = ("revenue","sum"),
#     margin_mean = ("margin_rate","mean"),
#     deals       = ("client_id","count"),
# ).reset_index().sort_values("revenue_sum", ascending=False)

# 8. СОХРАНЕНИЕ
# result.to_excel("result.xlsx", index=False)

print("Шаблон готов!")

---
## Шпаргалка (Quick Reference)

### Загрузка / Запись
```python
pd.read_csv("f.csv", sep=";", encoding="cp1251", usecols=[...])
pd.read_excel("f.xlsx", sheet_name="Лист1")
pd.read_parquet("f.parquet")
pd.read_sql(query, conn)

df.to_csv("f.csv", index=False, encoding="utf-8-sig")
df.to_excel("f.xlsx", index=False)
df.to_parquet("f.parquet")
```

### Просмотр
```python
df.head(n) / df.tail(n) / df.sample(n)
df.info()  / df.describe() / df.shape
df.dtypes  / df.columns / df.index
df["col"].value_counts() / .nunique() / .unique()
```

### Выбор данных
```python
df["col"]               # Series
df[["a","b"]]           # DataFrame
df.loc[rows, cols]      # по меткам
df.iloc[rows, cols]     # по позиции
df[df["col"] > 0]
df[(df["a"]>0) & (df["b"]<10)]   # AND
df[(df["a"]>0) | (df["b"]<10)]   # OR
df[~(df["a"]>0)]                  # NOT
df.query("a > 0 and b < 10")
df[df["col"].isin([...])]
df[df["col"].between(a, b)]
```

### Пропуски
```python
df.isna().sum()
df.dropna() / df.dropna(subset=["col"])
df.fillna(0) / df.fillna(df["col"].mean())
df.ffill() / df.bfill()
```

### Типы
```python
df.astype({"col": "int32"})
pd.to_numeric(s, errors="coerce")
pd.to_datetime(s, errors="coerce")
df["col"].astype("category")
```

### Условные колонки
```python
np.where(cond, a, b)
np.select([c1,c2], [v1,v2], default=v3)
pd.cut(col, bins=[...], labels=[...])
pd.qcut(col, q=4, labels=[...])
```

### Строки (.str)
```python
.str.lower() / .upper() / .title() / .strip()
.str.contains("x") / .startswith("x") / .endswith("x")
.str.replace("a","b", regex=False)
.str.split(",", expand=True)
.str.extract(r"pattern")
```

### Группировки
```python
df.groupby("col")["val"].sum()
df.groupby("col").agg(name=("col","func"))
df.groupby("col")["val"].transform("sum")
df.groupby("col").filter(lambda x: ...)
```

### Объединение
```python
pd.concat([df1,df2], ignore_index=True)   # вертикально
pd.concat([df1,df2], axis=1)              # горизонтально
df.merge(other, on="key", how="left")     # JOIN
# validate="many_to_one" / indicator=True
```

### Сводные
```python
pd.pivot_table(df, values, index, columns, aggfunc, fill_value, margins)
pd.crosstab(df["a"], df["b"], normalize="index")
```

### Reshape
```python
df.melt(id_vars, value_vars, var_name, value_name)  # wide->long
df.pivot(index, columns, values)                      # long->wide
df.stack() / df.unstack()
df.explode("col")
```

### Даты
```python
pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
.dt.year / .month / .day / .weekday / .quarter
.dt.strftime("%Y-%m")
.dt.to_period("M").dt.to_timestamp()
(date_end - date_start).dt.days
df.resample("ME").sum()
```

### Оконные функции
```python
.rolling(n).mean() / .sum() / .max()
.expanding().mean() / .sum()
.ewm(span=n).mean()
.shift(n) / .pct_change()
```

### Ранжирование
```python
.rank(ascending=False, method="dense")
.cumsum() / .cumcount()
groupby().cumsum() / groupby().cumcount()
```

### Цепочки
```python
df.assign(new=lambda d: d["a"] - d["b"])
df.pipe(func, **kwargs)
```

### Производительность
```python
df.memory_usage(deep=True).sum()
.astype("category") / .astype("uint8")
pd.to_numeric(s, downcast="integer")
pd.read_csv(..., usecols=[...], chunksize=100_000)
```

### Стилизация
```python
df.style.format({"col": "{:,.0f}"})
       .background_gradient(subset=["col"], cmap="RdYlGn")
       .highlight_max(color="#c6efce")
       .bar(subset=["col"], color="#d9e8fb")
       .set_caption("Заголовок")
```

---

## Источники

- 📖 [Официальная документация pandas](https://pandas.pydata.org/docs/)
- 📘 [User Guide](https://pandas.pydata.org/docs/user_guide/)
- 🔧 [API Reference](https://pandas.pydata.org/docs/reference/)
- 📚 Книга: *Python for Data Analysis* — Wes McKinney (создатель pandas)
